# Report Reader
Run the cells in order with the `train.csv` and `test.csv` beside the notebook, or set `DATA_DIR` below. The final cell writes **`report_reader_outputs/submission.csv`**.


In [1]:
from collections import defaultdict, Counter
from dataclasses import dataclass
from functools import partial
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.optimize import lsq_linear
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

SEED = 5
DATA_DIR = None  # Example: Path("data")
OUTPUT_FILE = Path("report_reader_outputs/submission.csv")
DOMAINS = ("Hospital", "Farmer", "Real Estate", "Corporate")

def find_data_dir(explicit_dir):
    if explicit_dir is not None:
        return Path(explicit_dir).expanduser()
    for folder in (Path.cwd(), Path("data"), Path("../data")):
        if all((folder / name).is_file() for name in ("train.csv", "test.csv")):
            return folder
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        pairs = sorted(p.parent for p in kaggle_input.rglob("train.csv")
                       if (p.parent / "test.csv").is_file())
        if len(pairs) == 1:
            return pairs[0]
        if len(pairs) > 1:
            raise ValueError(f"Set DATA_DIR to the Report Reader folder. Found {pairs}")
    raise FileNotFoundError("Set DATA_DIR to the folder containing train.csv and test.csv.")


def load_reports(folder):
    train = pd.read_csv(folder / "train.csv", dtype={"id": str})
    test = pd.read_csv(folder / "test.csv", dtype={"id": str})
    for frame, required in ((train, {"id", "text", "target"}), (test, {"id", "text"})):
        if not required.issubset(frame.columns):
            raise ValueError(f"Missing columns {required - set(frame.columns)} in {folder}")
        if frame.id.isna().any() or not frame.id.str.fullmatch(r"\d{5}").all() or not frame.id.is_unique:
            raise ValueError("Report IDs must be unique five-digit strings within each CSV.")
        if frame.text.isna().any() or not frame.text.str.strip().ne("").all():
            raise ValueError("Each report needs nonempty text.")
    if not np.isfinite(train.target.to_numpy(dtype=float)).all():
        raise ValueError("Training targets must all be finite.")
    return train, test


data_dir = find_data_dir(DATA_DIR)
train, test = load_reports(data_dir)
print(f"{len(train):,} labeled reports and {len(test):,} test reports")


1,195 labeled reports and 1,194 test reports


## Read amounts and fiscal years
The shared parsing functions normalize Indian and international currency units, number words, dates, and table cells. They retain labels needed to distinguish actuals, revisions, forecasts, and other entities.


In [2]:
# Shared currency and number normalization
ONES = {
    'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11,
    'twelve': 12, 'thirteen': 13, 'fourteen': 14, 'fifteen': 15,
    'sixteen': 16, 'seventeen': 17, 'eighteen': 18, 'nineteen': 19,
}

TENS = {
    'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50,
    'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90,
}

NUM_WORD = (
    r'(?:zero|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|'
    r'thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|'
    r'thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred|thousand)'
)

NUM_PHRASE_RE = re.compile(
    rf'\b({NUM_WORD}(?:(?:[\s\-]+(?:and\s+)?){NUM_WORD})*)\b'
    r'(?=\s*(?:crore|crores|cr\b|lakh|lakhs|million|billion|rupees|per\s+cent|'
    r'percent|%|towers?|floors?|residential\s+floors?|apartments?|units?|acres?|'
    r'quintals?|tonnes?|beds?|doctors?|offices?|inpatient|admissions|'
    r'per\s+(?:month|day|acre|quintal|unit|apartment|sq)|every\s+day|each\s+day|'
    r'each\s+month|a\s+day|a\s+month|sq\s*ft|square\s+feet|\|))',
    re.IGNORECASE,
)

TABLE_NUM_WORD_RE = re.compile(
    rf'(\|\s*)({NUM_WORD}(?:(?:[\s\-]+(?:and\s+)?){NUM_WORD})*)(\s*\|)',
    re.IGNORECASE,
)

COLON_NUM_WORD_RE = re.compile(
    rf'(:\s*)({NUM_WORD}(?:(?:[\s\-]+(?:and\s+)?){NUM_WORD})*)(?=\s*(?:[.,;\n]|$))',
    re.IGNORECASE,
)

UNIT_MULT = {
    'crore': 1e7, 'crores': 1e7, 'cr': 1e7,
    'lakh': 1e5, 'lakhs': 1e5, 'lac': 1e5, 'lacs': 1e5,
    'billion': 1e9, 'billions': 1e9, 'bn': 1e9,
    'million': 1e6, 'millions': 1e6, 'mn': 1e6,
    'thousand': 1e3,
}

MONEY_PATTERN = re.compile(
    r'(?:₹|\bRs\b\.?|\bINR\b)\s*(\d+(?:,\d+)*(?:\.\d+)?)\s*(crores?|cr\b|lakhs?|lacs?|billions?|bn\b|millions?|mn\b|thousand)?'
    r'|(\d+(?:,\d+)*(?:\.\d+)?)\s*(crores?|cr\b|lakhs?|lacs?|billions?|bn\b|millions?|mn\b)(?:\s*rupees)?'
    r'|(\d+(?:,\d+)*(?:\.\d+)?)\s+rupees',
    re.IGNORECASE,
)

def words_to_number(phrase: str):
    tokens = [t for t in re.split(r'[\s\-]+', phrase.lower().strip()) if t and t != 'and']
    if not tokens:
        return None
    total, current = 0, 0
    for tok in tokens:
        if tok in ONES:
            current += ONES[tok]
        elif tok in TENS:
            current += TENS[tok]
        elif tok == 'hundred':
            current = (current if current else 1) * 100
        elif tok == 'thousand':
            total += (current if current else 1) * 1000
            current = 0
        else:
            return None
    return total + current

def normalize_text(text: str) -> str:
    text = str(text).replace('–','-').replace('—','-').replace('−','-').replace('\u00a0',' ')
    text = re.sub(r'\bper\s+cent\b|\bpercent\b','%',text,flags=re.I)
    text = re.sub(r'\bRs\.\s*', 'Rs ', text)
    text = re.sub(r'\b[Cc][Rr]\.(?=\s+[a-z,;)])', 'Cr', text)
    text = re.sub(r'\bsq\.\s*ft\.?', 'sq ft', text, flags=re.IGNORECASE)

    def _repl(m):
        """Normalize one matched number phrase."""
        raw = m.group(1).lower().strip()
        if raw == 'one':
            return m.group(0)
        val = words_to_number(raw)
        return str(val) if val is not None else m.group(0)

    text = NUM_PHRASE_RE.sub(_repl, text)

    def _repl_tbl(m):
        """Normalize one matched table value."""
        val = words_to_number(m.group(2))
        return f"{m.group(1)}{val}{m.group(3)}" if val is not None else m.group(0)

    text = TABLE_NUM_WORD_RE.sub(_repl_tbl, text)

    def _repl_colon(m):
        """Normalize one colon-delimited value."""
        val = words_to_number(m.group(2))
        return f"{m.group(1)}{val}" if val is not None else m.group(0)

    text = COLON_NUM_WORD_RE.sub(_repl_colon, text)
    return text

def find_all_money(s: str):
    res = []
    for m in MONEY_PATTERN.finditer(s):
        if m.group(1) is not None:
            num = float(m.group(1).replace(',', ''))
            unit = (m.group(2) or '').lower()
        elif m.group(3) is not None:
            num = float(m.group(3).replace(',', ''))
            unit = (m.group(4) or '').lower()
        else:
            num = float(m.group(5).replace(',', ''))
            unit = ''
        res.append((m.start(), m.end(), num * UNIT_MULT.get(unit, 1.0), m.group(0)))
    return res


## Corporate reports
Build a ledger for the reporting company and derive operating profit for three consecutive fiscal years. Explicit dates take priority over inherited dates. An undated recap can be matched to a year when two financial metrics identify that year uniquely.

In [3]:
# Corporate entities, fiscal years and financial terms
CORPORATE_YEAR_RE = re.compile(
    r"(?P<fyr>\bFY\s?(?P<a1>20\d{2})\s?-\s?(?P<a2>\d{2})\b)"
    r"|(?P<rng>\b(?P<b1>20\d{2})\s?-\s?(?P<b2>\d{2})\b)"
    r"|(?P<yem>\byear\s+ended\s+(?:31\s+)?March,?\s+(?P<e>20\d{2})\b)"
    r"|(?P<fy4>\b(?:FY|fiscal(?:\s+year)?|financial\s+year)\s?(?P<c>20\d{2})\b)"
    r"|(?P<fy2>\bFY\s?'?(?P<d>\d{2})\b)"
    r"|(?P<bare>\b(?P<f>20\d{2})\b(?!\s*(?:crore|cr\b|lakh|million|billion|rupees|employees|people|beds|offices)))",
    re.I)

REL_FOLLOW = re.compile(
    r"\b(?:the\s+)?(?:following[\s-](?:year|fiscal(?:\s+year)?)|"
    r"(?:fiscal\s+)?year\s+(?:immediately\s+|right\s+|just\s+|directly\s+|straight\s+)?(?:after|following)|"
    r"year\s+that\s+(?:followed|came\s+(?:right\s+|immediately\s+|just\s+)?after)|next\s+year|subsequent\s+year|"
    r"succeeding\s+year)\b", re.I)
REL_PRECEDE = re.compile(
    r"\b(?:the\s+)?(?:preceding[\s-](?:year|fiscal(?:\s+year)?)|previous[\s-]year|prior[\s-]year|"
    r"(?:fiscal\s+)?year\s+(?:immediately\s+|right\s+|just\s+|directly\s+|straight\s+)?(?:before|preceding|prior\s+to)|"
    r"year\s+that\s+came\s+(?:right\s+|immediately\s+|just\s+)?before|(?:a|one)\s+year\s+(?:earlier|before)(?:\s+than)?|"
    r"year\s+earlier)\b", re.I)
REL_LATEST = re.compile(r"\b(?:the\s+)?(?:most\s+recent|latest|last\s+completed|latest\s+completed|most\s+recently\s+completed|just[\s-]ended|current)\s+(?:fiscal\s+|financial\s+|reporting\s+|completed\s+)?(?:year|period)\b", re.I)
REL_MIDDLE = re.compile(r"\b(?:middle\s+year|year\s+in\s+between|year\s+(?:that\s+(?:sits|falls|lies|came|comes)\s+)?(?:in\s+)?between|in-between\s+year|intervening\s+year|"
                        r"reference\s+year\s+in\s+between)\b", re.I)
SAME_RE = re.compile(r"\b(?:that\s+same\s+(?:year|period)|that\s+(?:year|period)|the\s+same\s+(?:year|period)|"
                     r"this\s+year|then)\b", re.I)
IT_ANCHOR = re.compile(r"^\s*(?:it|that(?!\s+year)|this)\b", re.I)
BASE_PREP = re.compile(r"(?:over|against|from|on|than|compared\s+(?:with|to)|versus|vs\.?|relative\s+to|"
                       r"above|below|beyond)\s*(?:the\s+level\s+of\s+|that\s+of\s+|the\s+figure\s+(?:for|of)\s+)?$", re.I)

FORECAST_RE = re.compile(r"guidance|guid(?:es?|ed|ing)\s+(?:for|to|towards|that|at|revenue|its|on)\b|(?:management|company|group|firm|board)\s+(?:has\s+)?guid|forecast|outlook|expects?\b|expected|projected|projection|"
                         r"\btarget(?:s|ed)?\s+(?:of|for|revenue)|budget(?:ed)?\s+(?:for|revenue)|plan(?:s|ned)?\s+(?:for\s+)?revenue|estimate[ds]?\s+(?:of|for|revenue)", re.I)
BASEYEAR_RE = re.compile(r"before\s+the\s+(?:period|review|years?\s+under)|base\s+(?:year|before|from|against)|"
                         r"the\s+base\b|sets\s+the\s+base|starting[\s-]point|reference\s+point|"
                         r"ahead\s+of\s+the\s+(?:period|review)|prior\s+to\s+the\s+(?:period|review)", re.I)
ORIG_RE = re.compile(r"(?:first|originally|initially|earlier|previously|provisionally)\s+(?:been\s+)?"
                     r"(?:reported|stated|published|announced|put|shown|filed|disclosed|recorded|given)"
                     r"(?:\s+(?:as|at|to\s+be|of))?\s*$|"
                     r"(?:original|earlier|initial|provisional|superseded)\s+(?:figure|number|amount|estimate)"
                     r"(?:\s+(?:of|was|stood\s+at|had\s+been))?\s*$|"
                     r"(?:we|it|the\s+company|management)\s+(?:first|originally|initially)\s+reported\s*$|"
                     r"reported\s+(?:at|as)\s*$", re.I)

METRIC_RE = re.compile(
    r"(?P<totexp>total\s+(?:expenses|expenditure|costs?)|wider\s+cost|broader\s+(?:cost|measure|expense))"
    r"|(?P<npat>net\s+profit|profit\s+after\s+tax|net\s+income|bottom[\s-]line|net\s+loss|PAT\b)"
    r"|(?P<mcap>market\s+(?:capitali[sz]ation|value|cap)|valuation)"
    r"|(?P<capex>capital\s+expenditure|capex|capital\s+spend|set\s+aside|earmark(?:ed|s)?|new\s+(?:facility|plant))"
    r"|(?P<opex>operating\s+(?:expenses?|costs?|expenditure)|opex|running\s+costs|cost\s+base|cost\s+line|"
    r"operating\s+cost\s+base|cost\s+(?:share|ratio|intensity|load|bill)|(?:expense|cost)[\s-]to[\s-](?:revenue|sales)|\b(?:the|that|this)\s+ratio\b|cost-to-revenue(?:\s+ratio)?|\b(?:spent|spending)\b(?=(?:(?!\.\s)[^;]){0,45}\bon\s+(?:operat|running)))"
    r"|(?P<op>operating\s+(?:profit|loss|income|surplus)|EBIT\b|(?:an?\s+)?operating\s+(?:deficit))"
    r"|(?P<rev>revenues?|top[\s-]?line|turnover|sales|booked|earned|income\s+from\s+operations)",
    re.I)

GROWTH_UP = r"rose|rise|risen|rising|grew|grow|growth|grown|increase[ds]?|increasing|climb(?:ed|s)?|up\b|higher|jump(?:ed)?|gain(?:ed)?|added|expan(?:ded|sion)|improv(?:ed|ement)|advance[ds]?|edged\s+up|lifted"
GROWTH_DN = r"fell|fall|fallen|falling|declin(?:e|ed|ing)|decrease[ds]?|dip(?:ped)?|slip(?:ped)?|slid|drop(?:ped)?|down\b|lower|contract(?:ed|ion)|shrank|eased|retreat(?:ed)?|weaken(?:ed)?"
GROWTH_RE = re.compile(rf"\b(?:(?P<up>{GROWTH_UP})|(?P<dn>{GROWTH_DN}))", re.I)
PCT_RE = re.compile(r"(-?\d+(?:\.\d+)?)\s*%")
RIVAL_WORD = re.compile(r"\b(?:rival|competitor|peer|competition|comparison\s+with|compared\s+with\s+(?:its\s+)?rival)\b", re.I)

COMPANY_RE = re.compile(r"\b([A-Z][a-z]+)\s+(Ltd|Corp|Group|Holdings|Industries|Enterprises|Technologies)\b")

def _year_of(m):
    g = m.groupdict()
    if g['fyr']:
        return 2000 + int(g['a2']) if int(g['a2']) != (int(g['a1']) + 1) % 100 else int(g['a1']) + 1
    if g['rng']:
        return int(g['b1']) + 1
    if g['yem']:
        return int(g['e'])
    if g['fy4']:
        return int(g['c'])
    if g['fy2']:
        return 2000 + int(g['d'])
    if g['bare']:
        return int(g['f'])
    return None

ENTITY_DETERMINERS = {"the","our","its","this","that","their"}

def entities(text):
    names = {}
    for m in COMPANY_RE.finditer(text):
        if m[1].casefold() in ENTITY_DETERMINERS:
            continue
        names.setdefault((m.group(1), m.group(2)), m.start())
    main = [n for n in names if n[1] != 'Corp']
    rivals = [n for n in names if n[1] == 'Corp']
    main = sorted(main, key=lambda n: names[n])
    main_words = {main[0][0]} if main else set()
    extra = {n[0] for n in main[1:]}
    rival_words = {n[0] for n in rivals} | extra
    rival_words -= main_words
    return main_words, rival_words



@dataclass(slots=True, eq=False)
class Tok:
    pos: int
    end: int
    kind: str
    val: object = None
    extra: object = None


@dataclass(slots=True)
class Fact:
    metric: str
    value: float
    year: object
    orig: bool
    forecast: bool
    base: bool
    order: int
    revised: bool
    binding: str = "context"
    source: str = ""
    rival: bool = False

def tokenize(s, main_words, rival_words):
    toks = []
    money = find_all_money(s)
    mspans = [(a, b) for a, b, _, _ in money]

    def in_money(a, b):
        return any(not (b <= x or a >= y) for x, y in mspans)

    for a, b, v, _ in money:
        toks.append(Tok(a, b, 'money', v))
    for m in PCT_RE.finditer(s):
        if not in_money(m.start(), m.end()):
            toks.append(Tok(m.start(), m.end(), 'pct', float(m.group(1))))
    occupied = []
    for rx, kind in ((REL_LATEST, 'latest'), (REL_MIDDLE, 'middle'), (REL_FOLLOW, 'follow'), (REL_PRECEDE, 'precede')):
        for m in rx.finditer(s):
            if any(not (m.end() <= x or m.start() >= y) for x, y in occupied):
                continue
            occupied.append((m.start(), m.end()))
            toks.append(Tok(m.start(), m.end(), 'rel', kind))
    for m in CORPORATE_YEAR_RE.finditer(s):
        if in_money(m.start(), m.end()):
            continue
        y = _year_of(m)
        if y is None or not (2005 <= y <= 2040):
            continue
        pre = s[max(0, m.start() - 40):m.start()].lower()
        if m.group('bare') and re.search(r'(?:founded|established|incorporated|registered|since|set\s+up|began\s+operations)\s+(?:in\s+)?$', pre):
            continue
        shift = 0
        pre2 = s[max(0, m.start() - 30):m.start()]
        if re.search(r'\b(?:before|preceding|precedes|preceded|prior\s+to|ahead\s+of|earlier\s+than)\s+(?:the\s+)?$', pre2, re.I):
            shift = -1
        elif re.search(r'\b(?:after|following|follows|followed|succeeding)\s+(?:the\s+)?$', pre2, re.I) and not re.search(r'\bin\s+the\s+(?:years?|period)\s+after\s+$', pre2, re.I):
            shift = 1
        toks.append(Tok(m.start(), m.end(), 'year', y + shift, shift))
    for m in SAME_RE.finditer(s):
        if any(not (m.end() <= x or m.start() >= y) for x, y in occupied):
            continue
        toks.append(Tok(m.start(), m.end(), 'same'))
    for m in METRIC_RE.finditer(s):
        kind = m.lastgroup
        txt = m.group(0).lower()
        loss = 'loss' in txt or 'deficit' in txt
        toks.append(Tok(m.start(), m.end(), 'metric', kind, loss))
    for w in main_words:
        for m in re.finditer(rf"\b{re.escape(w)}\b", s):
            toks.append(Tok(m.start(), m.end(), 'ent', 'main'))
    for w in rival_words:
        for m in re.finditer(rf"\b{re.escape(w)}\b", s):
            toks.append(Tok(m.start(), m.end(), 'ent', 'rival'))
    for m in re.finditer(r"\b(?:its|their|it)\b", s, re.I):
        toks.append(Tok(m.start(), m.end(), 'pron'))
    toks.sort(key=lambda t: (t.pos, -t.end))
    return toks

# Sentence and table readers and corporate fact representation
def split_sentences(block):
    p = re.sub(r'(\d)\.(\d)', r'\1<DEC>\2', block)
    p = re.sub(r'\b(Rs|No|vs|Ltd|Corp|approx|Mr|Ms|Dr|St)\.', r'\1<DOT>', p)
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z"\'\*\(\d₹])', p)
    return [x.replace('<DEC>', '.').replace('<DOT>', '.').strip() for x in parts if x.strip()]

GENERIC_HDR = re.compile(r"^(?:metric|item|measure|figure|amount|value|period|year|fiscal\s+year|financial\s+year|"
                         r"particulars|detail|details|line\s+item|company|entity|parameter|indicator|data|"
                         r"company\s+and\s+metric|figure\s+\(.*\)|amount\s+\(.*\))$", re.I)

def table_statements(rows):
    if not rows:
        return []
    hdr = rows[0]
    out = []
    if len(hdr) >= 2:
        year_cols = {j: h for j,h in enumerate(hdr) if FY_LABEL.fullmatch(h.strip())}
        if len(year_cols) >= 2 and set(year_cols) == set(range(1, len(hdr))):
            for row in rows[1:]:
                label = " ".join([hdr[0] if not GENERIC_HDR.fullmatch(hdr[0]) else "", row[0]]).strip()
                for j,period in year_cols.items():
                    if j < len(row) and (find_all_money(row[j]) or PCT_RE.search(row[j])):
                        out.append(f"{label} in {period}: {row[j]}")
            return out
    h0 = hdr[0] if hdr and not GENERIC_HDR.match(hdr[0].strip()) else ''
    for r in rows[1:]:
        labels = [h0] if h0 else []
        for j, c in enumerate(r):
            if not find_all_money(c) and not PCT_RE.search(c):
                labels.append(c)
        for j, c in enumerate(r):
            if find_all_money(c) or PCT_RE.search(c):
                h = hdr[j] if j < len(hdr) else ''
                hs = '' if GENERIC_HDR.match(h.strip()) else h
                lab = ' '.join(x for x in labels if x and not re.fullmatch(r'n/?a|-|—|not\s+shown|not\s+stated', x.strip(), re.I))
                out.append(f"{lab} {hs}: {c}".strip())
    return out

def corporate_units(text, table_fn=table_statements):
    lines = text.split('\n')
    para = 0
    i = 0
    out = []
    section = None
    while i < len(lines):
        s = lines[i].strip()
        if not s:
            para += 1
            i += 1
            continue
        if s.startswith('|'):
            rows = []
            while i < len(lines) and lines[i].strip().startswith('|'):
                ln = lines[i].strip()
                if not re.match(r'^\|[\s\-:|]+\|$', ln):
                    rows.append([c.strip() for c in ln.strip('|').split('|')])
                i += 1
            para += 1
            hdr_txt = ' '.join(rows[0]) if rows else ''
            for st in table_fn(rows):
                out.append(('table', st, para, section, hdr_txt))
            para += 1
            continue
        heading = None
        m = re.match(r'^(?:#+\s*(.+)|\*\*([^*]+)\*\*\s*$|\*([^*]+)\*\s*$)', s)
        if m:
            heading = next(g for g in m.groups() if g)
            out.append(('heading', heading.strip(), para, None, ''))
            para += 1
            i += 1
            continue
        m = re.match(r'^(\*\*[^*]+\*\*|\*[^*]+\*)\s*[:.\-]?\s*(.+)$', s)
        if m and len(m.group(1)) < 80:
            out.append(('heading', m.group(1).strip('*').strip(' .:'), para, None, 'inline'))
            s = m.group(2)
        kind = 'bullet' if re.match(r'^[-*•]\s', s) else 'q' if re.match(r'^Q\s*:', s) else 'a' if re.match(r'^A\s*:', s) else 'prose'
        s2 = re.sub(r'^(?:[-*•]\s+|[QA]\s*:\s*)', '', s)
        for sent in split_sentences(s2):
            out.append((kind, sent, para, None, ''))
        if kind in ('prose', 'a', 'q'):
            para += 1
        i += 1
    return out


def _resolve_rel(tok, toks, idx, s, ctx_year):
    nxt = s[tok.end:tok.end + 45]
    for t in toks[idx + 1: idx + 4]:
        if t.kind == 'year' and t.pos - tok.end <= 30 and re.match(r'^[\s,]*(?:(?:the|of|to|in|immediately|right|that\s+is|i\.e\.|namely)[\s,]+)*$', s[tok.end:t.pos], re.I):
            if t.extra:   # already shifted by its own before/after cue
                return t.val, t
            if tok.val == 'middle':
                m2 = re.match(r'^\s*(?:\S+\s+){0,3}?(?:and|to)\s+', s[t.end:t.end + 25])
                t_next = next((u for u in toks[idx + 2: idx + 6] if u.kind == 'year'), None)
                if m2 and t_next is not None and t_next.pos <= t.end + 30:
                    return (t.val + t_next.val) // 2, t
            if ',' in s[tok.end:t.pos] or tok.val in ('middle', 'latest'):   # apposition: "the middle year, 2021-22"
                return t.val, t
            if tok.val == 'follow':
                return t.val + 1, t
            if tok.val == 'precede':
                return t.val - 1, t
    if IT_ANCHOR.match(nxt) and isinstance(ctx_year, int) and tok.val in ('follow', 'precede') and \
            re.search(r'(?:after|before|following|preceding|prior\s+to|than)$', s[tok.pos:tok.end], re.I):
        return (ctx_year + 1 if tok.val == 'follow' else ctx_year - 1), 'it'
    return ('H', tok.val), None

DEFN_RE = re.compile(r"\s*\((?:i\.e\.\s*|that\s+is\s*|being\s+|meaning\s+|which\s+is\s+)?(?:revenue|sales)\s+(?:less|minus)\s+(?:operating\s+)?(?:expenses|costs)\)"
                     r"|,?\s*(?:being|meaning|i\.e\.|that\s+is|which\s+is|which\s+equals|defined\s+as|calculated\s+as|computed\s+as|measured\s+as)\s+"
                     r"(?:revenue|sales)\s+(?:less|minus|net\s+of|after(?:\s+deducting)?)\s+(?:operating\s+)?(?:expenses|costs)\s*,?"
                     r"|\b(?:revenue|sales)\s+(?:less|minus|net\s+of)\s+operating\s+(?:expenses|costs)\b"
                     r"|,?\s*(?:after|on)\s+(?:deducting|subtracting|netting\s+off)\s+operating\s+(?:expenses|costs)\s+from\s+(?:revenue|sales)\s*,?", re.I)

STRONG_BASE = re.compile(r"\b(?:over|against|compared\s+(?:with|to)|versus|vs\.?|than|relative\s+to|above|below)(?:\s+(?:the|that\s+of|that|this|its|revised|restated|earlier|original|prior|previous|level\s+of|levels?\s+(?:of|in)|figure\s+(?:for|of)))*$", re.I)
WEAK_BASE = re.compile(r"\b(?:on|from|above|below|beyond)(?:\s+(?:the|that\s+of|its|level\s+of|levels?\s+(?:of|in)))*$", re.I)

COMPLEMENT_RE = re.compile(r"(?:exceed(?:ed|ing|s)?|outran|outstripped|outpaced|against|relative\s+to|compared\s+(?:with|to))\s+(?:the\s+|that\s+(?:same\s+)?year.s\s+|its\s+|that\s+)?$|(?:%|share|proportion|fraction|percentage|percent)\s*(?:of|on)\s+(?:the\s+|that\s+year.s\s+|its\s+|this\s+|that\s+|annual\s+|the\s+year.s\s+|that\s+same\s+year.s\s+)?$", re.I)

def _is_complement(s, t):
    return bool(COMPLEMENT_RE.search(s[max(0, t.pos - 40):t.pos]))

CONTRAST_RE = re.compile(r"(?:not\s+(?:the\s+same\s+(?:thing\s+|measure\s+)?as|to\s+be\s+confused\s+with|be\s+read\s+as|counted\s+(?:as|in|within))|confused\s+with|different\s+(?:[\w-]+\s+){0,2}from|distinct\s+(?:[\w-]+\s+){0,2}from|rather\s+than|as\s+opposed\s+to|separate\s+(?:[\w-]+\s+){0,2}from|instead\s+of|unlike|apart\s+from)\s+(?:the\s+|its\s+|an?\s+)?(?:[\w-]+\s+){0,2}$", re.I)

MONEY_ATTACH_BASE = re.compile(r"\b(?:up|down|rising|falling|climbing|higher|lower|recovering)?\s*(?:from|against|versus|compared\s+(?:with|to))\s+(?:INR|Rs\.?|₹)\s*[\d.,]+\s*(?:crore|cr|lakh|million|billion)?\s+(?:in|for|during)\s+(?:the\s+)?$", re.I)

def _is_base(s, t, toks):
    pre = s[max(0, t.pos - 40):t.pos].rstrip()
    if STRONG_BASE.search(pre):
        return True
    if t.kind == 'rel' and re.search(r"\b(?:moved|rose|fell|grew|climbed|dipped|slipped|up|down|changed)\s+(?:\w+\s+){0,2}from\s+(?:the\s+)?(?:\w+\s+){0,3}(?:for|of|in)\s+(?:the\s+)?$", s[max(0, t.pos - 70):t.pos], re.I):
        return True
    if MONEY_ATTACH_BASE.search(s[max(0, t.pos - 70):t.pos]) and (
            re.search(r'\b(?:up|down|against|versus|compared)\b', s[max(0, t.pos - 70):t.pos], re.I) or
            re.match(r'^(?:(?!\.\s)[^;])*?\bto\s+(?:INR|Rs\.?|₹)', s[t.end:t.end + 120], re.I)):
        return True
    if t.kind == 'year' and (re.search(r'\b(?:from|between)\s+(?:the\s+)?$', s[max(0, t.pos - 15):t.pos], re.I) and
                             re.match(r'^\s*(?:to|through|until|and)\s+(?:the\s+)?(?:FY|fiscal|20\d\d|year)', s[t.end:t.end + 25], re.I)):
        return True
    if t.kind == 'year' and re.search(r'\bfrom\s+(?:FY\s?|fiscal\s+)?20\d\d(?:-\d\d)?\s+(?:to|through|until)\s+(?:the\s+)?$', s[max(0, t.pos - 40):t.pos], re.I):
        return True
    pre2 = s[max(0, t.pos - 70):t.pos]
    matches = list(re.finditer(r'\b(?:over|against|compared\s+(?:with|to)|versus|relative\s+to)\b', pre2, re.I))
    m = matches[-1] if matches else None
    if m is not None and not re.search(r'\d|\b(?:and|while|whereas|but)\b|[;,]', pre2[m.end():]) and PCT_RE.search(s[max(0, t.pos - 110):t.pos]):
        return True
    if WEAK_BASE.search(pre):
        win = s[max(0, t.pos - 90):t.pos]
        return bool(PCT_RE.search(win)) and bool(GROWTH_RE.search(win))
    return False

# Collect company facts and their reporting periods
def extract_facts(text, local_years=True):
    text = normalize_text(text)
    text = DEFN_RE.sub(' operating profit ', text)
    main_words, rival_words = entities(text)
    facts = []
    anchored = []
    explicit_years = set()
    forecast_years = set()
    base_years = set()
    sec_year = None
    sec_metric = None
    sec_rival = False
    sec_base = False
    para_year = None
    last_para = None
    last_entity = 'main'
    last_metric = None
    q_rival = False
    forecast_section = False
    last_actual_year = anchor_para = None
    order = 0
    us = corporate_units(text)
    for kind, s, para, _, _ in us:
        toks = tokenize(s, main_words, rival_words)
        if kind == 'heading':
            hl = s.lower()
            forecast_section = bool(re.fullmatch(r"(?:management |company )?(?:forecast|guidance|outlook|projections?)(?: (?:for )?(?:FY.*|20\d{2}.*))?[:.]?", s.strip(), re.I))
            is_rival = bool(RIVAL_WORD.search(s)) or any(t.kind == 'ent' and t.val == 'rival' for t in toks) or \
                bool(re.search(r'\bpeer|compar|competit', hl))
            mainment = any(t.kind == 'ent' and t.val == 'main' for t in toks)
            sec_rival = is_rival and not mainment
            sec_base = bool(BASEYEAR_RE.search(s)) or bool(re.search(r'earlier\s+reference|before\s+the\s+period', hl))
            sec_year = None
            sec_metric = None
            hm = [t.val for t in toks if t.kind == 'metric']
            if len(set(hm)) == 1:
                sec_metric = hm[0]
            for j, t in enumerate(toks):
                if t.kind == 'year':
                    sec_year = t.val
                    if t.extra:
                        anchored.append(('follow' if t.extra > 0 else 'precede', t.val))
                    elif not sec_rival and not FORECAST_RE.search(s):
                        explicit_years.add(t.val)
                    break
                if t.kind == 'rel':
                    sec_year, anc = _resolve_rel(t, toks, j, s, None)
                    if anc is not None and isinstance(sec_year, int):
                        anchored.append((t.val, sec_year))
                    break
            if sec_base and isinstance(sec_year, int):
                base_years.add(sec_year)
            if FORECAST_RE.search(s) and isinstance(sec_year, int):
                forecast_years.add(sec_year)
            para_year = sec_year
            last_para = para
            last_metric = None
            continue
        if kind == 'q':
            q_rival = bool(re.search(r'\b(?:rivals?|competitors?|peers?|competition)\b', s, re.I)) or any(t.kind == 'ent' and t.val == 'rival' for t in toks)
            for j, t in enumerate(toks):
                if t.kind == 'year':
                    para_year = t.val
                    if t.extra:
                        anchored.append(('follow' if t.extra > 0 else 'precede', t.val))
                    elif not q_rival and not FORECAST_RE.search(s):
                        explicit_years.add(t.val)
                    break
                if t.kind == 'rel':
                    y, anc = _resolve_rel(t, toks, j, s, para_year)
                    if not BASE_PREP.search(s[:t.pos]):
                        para_year = y
                        if anc is not None and isinstance(y, int):
                            anchored.append((t.val, y))
                    break
            continue
        if para != last_para:
            if kind != 'a':
                q_rival = False
            para_year = para_year if para_year is not None else sec_year
            last_para = para
            last_metric = None if kind != 'a' else last_metric
        sent_forecast = bool(FORECAST_RE.search(s)) or (forecast_section)
        if local_years and kind == "prose" and last_actual_year is not None and para == anchor_para:
            # A new clause starting with an anaphoric year is relative to the last stated period.
            rel = re.match(r"^(?:in |during |for )?(?:the )?(following|next|preceding|previous) (?:fiscal )?year(?=\s*[,;:]|\s+(?:revenue|sales|operating))",s,re.I)
            if rel and not any(t.kind == "year" for t in toks) and not re.search(r"(?im)^\|.*\b" + rel[1] + r" (?:fiscal )?year\b", text):
                step = 1 if rel[1].lower() in ("following","next") else -1
                s = f"FY{last_actual_year + step}" + s[rel.end():]
                toks = tokenize(s, main_words, rival_words)

        sent_base = bool(BASEYEAR_RE.search(s)) or sec_base
        cur_year = para_year
        sent_years = []
        prev_value_end = 0
        for j, t in enumerate(toks):
            if t.kind in ('year', 'rel', 'same'):
                is_base = _is_base(s, t, toks)
                if t.kind == 'year':
                    y = t.val
                    if t.extra and not is_base:
                        anchored.append(('follow' if t.extra > 0 else 'precede', y))
                    if not t.extra and not sent_forecast:
                        explicit_years.add(y)
                    if j > 0 and toks[j - 1].kind == 'rel' and t.pos - toks[j - 1].end <= 30 and \
                            re.match(r'^[\s,]*(?:(?:the|of|to|in|immediately|right|that\s+is|i\.e\.|namely)[\s,]+)*$', s[toks[j - 1].end:t.pos], re.I):
                        continue
                elif t.kind == 'rel':
                    y, anc = _resolve_rel(t, toks, j, s, cur_year)
                    if anc is not None and isinstance(y, int):
                        anchored.append((t.val, y))
                else:
                    y = cur_year
                t.extra = (y, is_base)
                if not is_base:
                    cur_year = y
                    sent_years.append(y)
                continue
        cur_metric = None
        cur_loss = False
        _fv = next((t.pos for t in toks if t.kind in ('money', 'pct')), len(s))
        _pron_rival = last_entity == 'rival' and not any(t.kind == 'ent' and t.pos < _fv for t in toks) and \
            bool(re.search(r'\b(?:it|its|their|they)\b', s[:_fv], re.I))
        ent_cur = 'rival' if (kind == 'a' and q_rival) or sec_rival or _pron_rival or re.match(r'^(?:peer\s+comparison|rival|competitor|comparison)', s, re.I) else 'main'
        if kind == 'table' and not sec_rival:
            ent_cur = 'main'
        cur_year = para_year
        sent_metric_seen = None
        from_metric = None
        for j, t in enumerate(toks):
            if t.kind == 'ent':
                if t.val == 'main' and re.search(r'(?:rival|competitor|peer)\s+(?:to|of)\s+(?:the\s+)?$|\bthan\s+$', s[max(0, t.pos - 30):t.pos], re.I):
                    continue
                ent_cur = t.val
                continue
            if t.kind == 'metric':
                if t.val == 'rev' and _is_complement(s, t):
                    continue
                if t.val in ('opex', 'rev', 'op') and CONTRAST_RE.search(s[max(0, t.pos - 45):t.pos]):
                    continue
                cur_metric = t.val
                cur_loss = t.extra
                sent_metric_seen = t.val
                continue
            if t.kind in ('year', 'rel', 'same') and t.extra is not None and isinstance(t.extra, tuple):
                y, is_base = t.extra
                if not is_base:
                    cur_year = y
                continue
            if t.kind not in ('money', 'pct'):
                continue
            metric = cur_metric
            loss = cur_loss
            if metric is None and last_metric is not None and re.search(r'\b(?:it|they)\s+(?:had\s+|has\s+|have\s+)?\w+', s[:t.pos], re.I) \
                    and not any(t2.kind == 'metric' for t2 in toks[:j]):
                metric = last_metric
            if metric is None:
                for t2 in toks[j + 1:]:
                    if t2.kind in ('money', 'pct'):
                        break
                    if t2.kind == 'metric' and not (t2.val == 'rev' and _is_complement(s, t2)):
                        metric, loss = t2.val, t2.extra
                        break
            if metric is None:
                metric = last_metric if last_metric is not None else sec_metric
            if t.kind == 'money':
                _lab = re.search(r"\b(?P<rev>revenues?|sales|turnover)\s+(?:of|was|were|at|stood\s+at|came\s+to|reached)\s*$|\b(?P<opex>operating\s+(?:expenses|costs))\s+(?:of|were|was|at|stood\s+at|came\s+to)\s*$",
                                 s[max(0, t.pos - 40):t.pos], re.I)
                if _lab and not CONTRAST_RE.search(s[max(0, t.pos - 85):_lab.start() + max(0, t.pos - 40)]):
                    metric = _lab.lastgroup
                    loss = False
            for t2 in toks[j + 1:j + 3]:
                if t.kind == 'money' and t2.kind == 'metric' and not _is_complement(s, t2) and re.match(r'^\s*(?:of|for)\s+(?:the\s+|its\s+)?$', s[t.end:t2.pos], re.I):
                    metric, loss = t2.val, t2.extra
                    break
            if t.kind == 'money' and from_metric is not None and re.search(r"(?:,|\bthen)\s+to\s*$", s[max(0, t.pos - 12):t.pos]) and metric != from_metric:
                metric, loss = from_metric, False
            if t.kind == 'money' and re.search(r"\bfrom\s*$", s[max(0, t.pos - 10):t.pos]):
                from_metric = metric
            prefix = s[max(0, t.pos - 60):t.pos]
            span = s[prev_value_end:t.pos]
            _oc = [m.end() for m in re.finditer(
                r'(?:first|originally|initially|earlier|previously|provisionally)\s+(?:been\s+)?(?:reported|stated|published|announced|put|shown|filed|disclosed|recorded|given|booked)|'
                r'(?:original|initial|provisional|earlier)\s+(?:figure|number|amount|estimate|version)\s+(?:of|was)|\boriginal\b|\bsuperseded\b(?!\s+by)', span, re.I)
                   if m.group(0).lower() not in ('original', 'superseded') or not re.search(r'(?:replaces|replacing|supersedes|superseding|in\s+place\s+of|instead\s+of)\s+(?:the\s+|that\s+|an?\s+|its\s+)?(?:\w+\s+)?\w*$', span[:m.start()], re.I)]
            _rc = [m.end() for m in re.finditer(r'revised|restated|corrected|replaced\s+by|superseded\s+by|now\s+stands|replaces\s+it|amended', span, re.I)]
            orig = bool(ORIG_RE.search(prefix.rstrip())) or (bool(_oc) and (not _rc or max(_oc) > max(_rc)))
            revised = bool(re.search(r'(?:revised|restated|corrected|replaced|superseded\s+by|amended|updated)(?:\s+\w+){0,5}\s+(?:to|is|was|of|at|by)?\s*$', prefix, re.I)) or \
                bool(re.search(r'^\W{0,3}(?:\s*\w+){0,4}\s+(?:replaces|supersedes)', s[t.end:t.end + 60], re.I)) or \
                bool(re.search(r'(?:corrected|revised|restated)\s+(?:\w+\s+){0,3}$', prefix, re.I))
            year = None
            binding = "context"
            for t2 in toks[j + 1:]:
                if t2.kind in ('money', 'pct'):
                    break
                if t2.kind == 'metric' and not (t2.val == 'rev' and _is_complement(s, t2)) and \
                        not re.match(r'^\s*(?:of|for)\s+(?:the\s+|its\s+)?$', s[t.end:t2.pos], re.I):
                    break
                if t2.kind in ('year', 'rel', 'same') and isinstance(t2.extra, tuple):
                    gap = s[t.end:t2.pos]
                    gap = re.sub(r'^\s*(?:of|for)\s+(?:the\s+|its\s+)?(?:revenues?|sales|operating\s+(?:expenses|costs|profit))\s+', ' ', gap, flags=re.I)
                    if (re.match(r'^\s*,?\s*(?:(?:in|for|during|of|by|at|the|end\s+of|close\s+of|across)\s+){0,3}$', gap, re.I) or
                            (re.match(r'^[^.;,]{0,70}?\b(?:in|for|during)\s+(?:the\s+)?$', gap, re.I) and not re.search(r'\b(?:and|while|whereas|but|against|compared)\b', gap, re.I))):
                        year = t2.extra[0]
                        binding = "explicit" if t2.kind == "year" else "relative"
                    break
            if year is None:
                for t2 in reversed(toks[:j]):
                    if t2.kind in ('year', 'rel', 'same') and isinstance(t2.extra, tuple):
                        y2, b2 = t2.extra
                        if b2 and not re.match(r'^\s*(?:that|it|this)?\s*,?\s*(?:when|in\s+which|where|during\s+which|whose)\b', s[t2.end:t.pos], re.I):
                            continue
                        year = y2
                        binding = "explicit" if t2.kind == "year" else "relative"
                        break
            if year is None:
                for t2 in toks[j + 1:]:
                    if t2.kind in ('year', 'rel') and isinstance(t2.extra, tuple) and not t2.extra[1]:
                        year = t2.extra[0]
                        binding = "explicit" if t2.kind == "year" else "relative"
                        break
            if year is None:
                year = para_year
            fc = sent_forecast and bool(FORECAST_RE.search(s[max(0, t.pos - 150):t.end + 80]))
            if fc and isinstance(year, int):
                forecast_years.add(year)
            if ent_cur != 'main':
                prev_value_end = t.end
                continue
            if forecast_section:
                fc = True
            if ent_cur == 'main' and not fc and isinstance(year, int) and binding != "context":
                last_actual_year, anchor_para = year, para
            if t.kind == 'pct':
                g = None
                for gm in GROWTH_RE.finditer(s[max(prev_value_end, t.pos - 90):t.pos]):
                    g = gm
                if g is None:
                    for gm in GROWTH_RE.finditer(s[max(0, t.pos - 120):t.pos]):
                        g = gm
                if g is None:
                    g = GROWTH_RE.search(s[t.end:t.end + 40])
                if metric in ('opex',) and not re.search(r'\b(?:rose|grew|fell|increased|decreased|declined|climbed|jumped|dropped)\s+(?:by\s+)?$', s[max(0, t.pos - 25):t.pos], re.I):
                    facts.append(Fact(metric='opex_pct', value=t.val, year=year, orig=orig, forecast=fc,
                                      base=sent_base, order=order, revised=revised, binding=binding, source=s, rival=(ent_cur != "main")))
                elif metric in ('rev', None) and g is not None:
                    sign = -1 if g.group('dn') else 1
                    gy = None
                    for t2 in toks[j + 1:]:
                        if t2.kind in ('money', 'pct'):
                            break
                        if t2.kind in ('year', 'rel', 'same') and isinstance(t2.extra, tuple):
                            y2, b2 = t2.extra
                            if b2:
                                if (isinstance(y2, tuple) and y2[0] == 'H' and t2.kind == 'rel') or t2.kind == 'same':
                                    if gy is None:
                                        gy = ('subj', cur_year if cur_year is not None else para_year)
                                else:
                                    gy = ('base', y2)
                                break
                            elif gy is None:
                                gy = ('subj', y2)
                    if gy is None:
                        for t2 in reversed(toks[:j]):
                            if t2.kind in ('year', 'rel', 'same') and isinstance(t2.extra, tuple):
                                y2, b2 = t2.extra
                                if b2 and ((t2.kind == 'rel' and isinstance(y2, tuple) and y2[0] == 'H') or t2.kind == 'same'):
                                    ctx = None
                                    for t3 in reversed(toks[:toks.index(t2)]):
                                        if t3.kind in ('year', 'rel', 'same') and isinstance(t3.extra, tuple) and not t3.extra[1]:
                                            ctx = t3.extra[0]
                                            break
                                    gy = ('subj', ctx if ctx is not None else para_year)
                                else:
                                    gy = ('base', y2) if b2 else ('subj', y2)
                                break
                    if gy is None:
                        gy = ('subj', para_year)
                    facts.append(Fact(metric='growth', value=sign * t.val, year=gy, orig=orig, forecast=fc,
                                      base=sent_base, order=order, revised=revised, binding=binding, source=s, rival=(ent_cur != "main")))
                order += 1
                prev_value_end = t.end
                continue
            if metric in ('rev', 'opex', 'op'):
                v = t.val / 1e7
                if metric == 'op' and loss:
                    v = -v
                if ent_cur == 'main':
                    facts.append(Fact(metric=metric, value=v, year=year, orig=orig, forecast=fc,
                                      base=sent_base, order=order, revised=revised, binding=binding, source=s, rival=(ent_cur != "main")))
            order += 1
            prev_value_end = t.end
        last_entity = ent_cur
        if sent_metric_seen:
            last_metric = sent_metric_seen
        if sent_years:
            para_year = sent_years[0] if len(set(map(str, sent_years))) == 1 else sent_years[-1]
        if sent_base:
            for y in sent_years:
                if isinstance(y, int):
                    base_years.add(y)
    return facts, (anchored, explicit_years), forecast_years, base_years

# Resolve operating profits in three consecutive fiscal years
def solve_corporate_years(facts, year_context, forecast_years, base_years):
    facts = [f for f in facts if not f.orig]
    def yrs(f):
        y = f.year
        if isinstance(y, tuple) and y and y[0] in ('base', 'subj'):
            y = y[1]
        return y
    E = sorted({yrs(f) for f in facts if isinstance(yrs(f), int) and not f.forecast})
    Efc = [y for y in E if y not in forecast_years or any(not f.forecast and yrs(f) == y and f.metric in ('opex', 'op', 'opex_pct') for f in facts)]
    E = Efc
    anchored, explicit_years = year_context
    E = [y for y in E if y in explicit_years] or E
    anchored = [(k, a) for k, a in anchored if a not in forecast_years]
    _Enb = [y for y in E if y not in base_years] or E
    run3 = any(all(y + k in _Enb for k in range(3)) for y in _Enb)
    _known = set(E) | {a for _, a in anchored}
    latest = (max(_known) if (run3 or any(a > max(E) for a in _known)) else max(E) + 1) if E else None
    Hk = {}
    gap = None
    if E:
        gaps = [y for y in range(min(E), max(E) + 1) if y not in E]
        if len(gaps) == 1 and max(E) - min(E) <= 3:
            gap = gaps[0]
    all_anch = [a for _, a in anchored]
    for kd in ('follow', 'precede', 'middle'):
        own = [a for k, a in anchored if k == kd]
        if own:
            Hk[kd] = Counter(own).most_common(1)[0][0]
        elif gap is not None:
            Hk[kd] = gap
        elif all_anch and kd != 'middle' and (
                (kd == 'follow' and Counter(all_anch).most_common(1)[0][0] > max(E or [0])) or
                (kd == 'precede' and Counter(all_anch).most_common(1)[0][0] < min(E or [9999]))):
            Hk[kd] = Counter(all_anch).most_common(1)[0][0]
        elif E:
            Hk[kd] = max(E) + 1 if kd == 'follow' else min(E) - 1 if kd == 'precede' else (min(E) + max(E)) // 2

    def ry(y):
        if isinstance(y, tuple):
            if y[0] == 'H':
                if y[1] == 'latest':
                    return latest
                return Hk.get(y[1])
            if y[0] in ('base', 'subj'):
                inner = ry(y[1])
                if inner is None:
                    return None
                return inner + 1 if y[0] == 'base' else inner
        return y

    # A relative summary can repeat two already dated metrics from the same period.
    # Rebind only a uniquely matched pair. Explicitly dated equal results remain independent.
    dated, summaries, rebound = defaultdict(set), defaultdict(list), {}
    for f in facts:
        if f.forecast or f.rival or f.metric not in ("rev","opex","op"):
            continue
        if f.binding == "explicit" and isinstance(f.year,int):
            dated[f.metric, f.value].add(f.year)
        if f.binding == "context" and isinstance(f.year,tuple) and f.year[0] == "H":
            summaries[f.source, f.year].append(f)
    for group in summaries.values():
        if len({f.metric for f in group}) < 2:
            continue
        match = set.intersection(*(dated[f.metric, f.value] for f in group))
        if len(match) == 1:
            year = next(iter(match))
            rebound.update({f.order:year for f in group if ry(f.year) != year})
    led = {}
    used = set()
    for f in facts:
        y = rebound.get(f.order, ry(f.year))
        if y is None or f.forecast or ((f.rival or f.order in used)):
            continue
        used.add(f.order)
        key = (f.metric, y)
        prev = led.get(key)
        if prev is not None and prev.binding == "explicit" and f.binding == "context" and not f.revised:
            continue
        if prev is None or (f.revised and not prev.revised) or (not prev.revised and f.order > prev.order and not f.base):
            led[key] = f
    years_all = sorted({k[1] for k in led})
    real = [y for y in years_all if y not in forecast_years or any(k[1] == y and k[0] in ('opex', 'op', 'opex_pct') for k in led)]
    if not real:
        return np.full(3, np.nan)
    def has_any(y):
        return any(k[1] == y and k[0] in ('opex', 'op', 'opex_pct', 'growth', 'rev') for k in led)
    Y3 = max(y for y in real if has_any(y))
    W = [Y3 - 2, Y3 - 1, Y3]

    rev_cache = {}

    def rev(y, depth=0):
        if y in rev_cache:
            return rev_cache[y]
        r = None
        if ('rev', y) in led:
            r = led[('rev', y)].value
        elif depth < 4 and ('growth', y) in led:
            p = rev(y - 1, depth + 1)
            if p is not None:
                r = p * (1 + led[('growth', y)].value / 100)
        if r is None and depth < 4 and ('growth', y + 1) in led and ('rev', y + 1) in led:
            r = led[('rev', y + 1)].value / (1 + led[('growth', y + 1)].value / 100)
        if r is None and ('opex', y) in led and ('opex_pct', y) in led and led[('opex_pct', y)].value > 0:
            r = led[('opex', y)].value / (led[('opex_pct', y)].value / 100)
        if r is None and ('opex', y) in led and ('op', y) in led:
            r = led[('opex', y)].value + led[('op', y)].value
        rev_cache[y] = r
        return r

    out = []
    for y in W:
        if ('op', y) in led:
            out.append(led[('op', y)].value)
            continue
        r = rev(y)
        if r is not None and ('opex', y) in led:
            out.append(r - led[('opex', y)].value)
        elif r is not None and ('opex_pct', y) in led:
            out.append(r * (1 - led[('opex_pct', y)].value / 100))
        else:
            out.append(np.nan)
    return np.array(out, dtype=float)


def parse_corporate(text):
    x = solve_corporate_years(*extract_facts(text))
    if not np.isfinite(x).all():
        # An ambiguous relative label can describe the same reference period across several clauses.
        # Prefer a full three-year ledger when document-wide anchors supply the missing years.
        alternative = solve_corporate_years(*extract_facts(text, local_years=False))
        if np.isfinite(alternative).sum() > np.isfinite(x).sum():
            return alternative
    return x


## Hospital reports
Extract inpatient revenue and annual admissions. Fiscal-year labels stay attached to their table rows. When a comparison gives two counts, the year and admissions context determine which count belongs to the reporting period. Monthly and daily rates are annualized.


In [4]:
@dataclass(slots=True)
class Quantity:
    role: str
    value: float
    orig: bool = False
    revised: bool = False
    prior: bool = False
    launch: bool = False
    current: bool = False


# Parse tables, rank candidate quantities and recognize revisions
ORIG_CUE = re.compile(r"(?:first|originally|initially|earlier|previously|provisionally)\s+(?:been\s+)?"
                      r"(?:reported|stated|put|given|listed|recorded|announced|published|shown|estimated|filed|circulated|quoted|released|disclosed|noted|indicated|calculated|computed|tallied|logged|cited)|"
                      r"\b(?:original|initial|provisional)\s+(?:figure|number|count|estimate|version|value|report|reporting)|"
                      r"\bsuperseded\b(?!\s+by)|\bas\s+first\s+reported|\bdisregard|\bfirst\s+(?:\w+\s+){1,3}(?:was|were)\s+(?:put|reported|given|stated|recorded|estimated|placed)\b|\breported\s+(?:earlier|originally|initially|first)\b|\b(?:early|earlier|initial|first|previous|prior)\s+(?:reports?|estimates?|filings?|disclosures?)\b|"
                      r"\b(?:original|initial)\s+(?:\w+\s+){0,2}(?:price|rate|figure|number|count|estimate|value|total)\s+(?:of|was|at|stood)\b|"
                      r"\b(?:earlier|previous|prior)\s+(?:figure|number|estimate|value|count)\b(?=[^.;]{0,40}(?:reported|stated|given|was|had))", re.I)
REV_CUE = re.compile(r"revised|restated|corrected|replaced\s+by|superseded\s+by|now\s+stands|updated|amended|\brevision\b|\breconcil\w*|"
                     r"\breplaces\b|\bcorrect\s+(?:figure|count|number|value)|\btrue\s+(?:figure|count|number)", re.I)
NUM_RE = re.compile(r"(?<![\w.₹/-])(\d{1,3}(?:,\d{2,3})+|\d+)(?:\.(\d+))?(?![\d,]*\s*(?:crore|cr\b|lakh|million|billion|rupees|%))")

def pre_normalize(text):
    def _r(m):
        v = float(m.group(1).replace(',', '')) * 1000
        return f"{v:,.0f}" if abs(v - round(v)) < 1e-9 else f"{v:,.2f}"
    return re.sub(r"(?<![\w.])(\d[\d,]*(?:\.\d+)?)\s+thousand\b", _r, text, flags=re.I)

def num(m):
    """Convert a numeric match to float."""
    return float(m.group(0).replace(',', ''))

FY_LABEL = re.compile(r"\bFY\s*['’]?\d{2,4}(?:\s*[-–/]\s*\d{2,4})?\b|\b(?:19|20)\d{2}(?:\s*[-–/]\s*\d{2,4})?\b", re.I)

def is_val(c):
    c = re.sub(r"\bFY\s*['’]?\d{2,4}(?:\s*[-–/]\s*\d{2,4})?\b", " ", c, flags=re.I)
    if re.search(r'₹|\bRs\b|\bINR\b|rupees|crore|lakh|%|per\s+cent', c, re.I) and re.search(r'\d', c):
        return True
    c2 = re.sub(r'\b(?:FY\s*)?(?:19|20)\d\d(?:\s*-\s*\d\d)?\b', ' ', c)
    if not re.search(r'\d', c2):
        return False
    words = [w for w in re.findall(r'[A-Za-z]{3,}', c2)
             if w.lower() not in ('crore', 'lakh', 'million', 'billion', 'rupees', 'per', 'quintal', 'quintals', 'acre', 'acres',
                                  'tonnes', 'tonne', 'square', 'feet', 'foot', 'unit', 'apartment', 'apartments', 'inr')]
    return len(words) <= 3

def table_statements_all(rows):
    if not rows:
        return []
    hdr = rows[0]
    out = []
    h0 = hdr[0] if hdr and not GENERIC_HDR.match(hdr[0].strip()) else ''
    KEY = r"\s*(?:measure|metric|indicator|item|particulars?|parameter)\s*"
    VAL = r"\s*(?:value|figure|amount|count|number)\s*"
    allrows = [hdr] + rows[1:]
    if len(hdr) == 2 and any(re.fullmatch(KEY, r[0], re.I) for r in allrows) and any(re.fullmatch(VAL, r[0], re.I) for r in allrows):
        out2 = []
        for k in range(len(allrows) - 1):
            a0, b0 = allrows[k], allrows[k + 1]
            if len(a0) == 2 and len(b0) == 2 and re.fullmatch(KEY, a0[0], re.I) and re.fullmatch(VAL, b0[0], re.I):
                out2.append(f"{a0[1]}: {b0[1]}")
        others = [r for r in allrows if not (re.fullmatch(KEY, r[0], re.I) or re.fullmatch(VAL, r[0], re.I))]
        for r in others:
            if len(r) == 2 and is_val(r[1]) and not is_val(r[0]):
                out2.append(f"{r[0]}: {r[1]}")
        if out2:
            return out2
    for r in rows[1:]:
        if r and ((is_val(r[0]) and sum(is_val(x) for x in r) >= 2) or
                  (re.fullmatch(r"\s*(?:value|figure|amount|count|number|total)\s*", r[0], re.I) and len(rows) == 2 and hdr and
                   re.fullmatch(r"\s*(?:measure|metric|item|indicator|particulars?)\s*", hdr[0], re.I))):
            for j, cc in enumerate(r):
                if is_val(cc):
                    out.append(f"{hdr[j] if j < len(hdr) else ''}: {cc}".strip())
            continue
        labels = [h0] if h0 else []
        labels += [x for j2, x in enumerate(r) if j2 > 0 and not is_val(x)]
        if r and not is_val(r[0]):
            labels.append(r[0])
        only_first = is_val(r[0]) and not any(is_val(x) for x in r[1:])
        for j, c in enumerate(r):
            if (is_val(c) and j > 0) or (j == 0 and (len(r) == 1 or only_first)):
                h = hdr[j] if j < len(hdr) else ''
                hs = '' if GENERIC_HDR.match(h.strip()) else h
                lab = ' '.join(x for x in labels if x and not re.fullmatch(r'n/?a|-|—|not\s+shown|not\s+stated', x.strip(), re.I))
                out.append(f"{hs} {lab}: {c}".strip())
    return out

def report_units(text):
    return corporate_units(text, table_fn=table_statements_all)

def pick_quantity(qs, role, prefer_current=False, exclude_prior=False):
    """Select the preferred candidate value."""
    candidates = [
        q for q in qs
        if q.role == role
        and not q.orig
        and (not exclude_prior or not q.prior)
    ]
    if prefer_current:
        candidates = [q for q in candidates if not q.launch]
    if not candidates:
        return None
    revised = [q for q in candidates if q.revised]
    if revised:
        return revised[-1].value
    if prefer_current:
        current = [q for q in candidates if q.current]
        if current:
            return current[0].value
    return candidates[0].value

def orig_revised(pre, post=''):
    oc = [m.end() for m in ORIG_CUE.finditer(pre)
          if not (re.search(r"(?:replaces|replacing|supersedes|in\s+place\s+of)\s+(?:the\s+|that\s+|an?\s+)?\w*\s*$", pre[:m.start()], re.I)
                  and re.match(r"^\s*\w*\s*\)", pre[m.end():]))]
    rc = [m.end() for m in REV_CUE.finditer(pre)]
    orig = bool(oc) and (not rc or max(oc) > max(rc))
    if orig and re.search(r"\bcurrent\b|\bat\s+present\b|\bnow\b|\btoday\b|:\s", pre[max(oc):]):
        orig = False
    revised = (bool(rc) and (not oc or max(rc) > max(oc))) or bool(re.match(r"^[^.;]{0,40}\b(?:replaces|supersedes)\b", post, re.I))
    return orig, revised

REVISED_TO = re.compile(r"(?:revised|restated|corrected|updated|changed|amended|raised|lowered|cut)(?:\s+\w+){0,4}\s+(?:to|at)\s*$|"
                        r"\b(?:revision|correction|restatement)\s+(?:to|of)\s*$|"
                        r"(?:revised|corrected|restated|correct|true|actual|verified)\s+(?:\w+\s+)?(?:figure|number|count|total|value)\s+(?:is|of|was|stands\s+at|to\s+be|at|as)\s*$|"
                        r"(?:corrected|revised|restated)\s+(?:figure|number|count|value)?\s*$", re.I)

def clauses(s):
    parts = re.split(r";\s+|\s+(?=of\s+which\b)|,\s+(?=and\b)|\s+(?=while\b|whereas\b)|,\s+(?=(?:as\s+)?against\b|compared\s+(?:with|to)\b|versus\b|vs\.?\s|well\s+(?:above|below)\b|ahead\s+of\b|behind\b)", s)
    return [p for p in parts if p.strip()]

pick_current_period = partial(pick_quantity, exclude_prior=True)

# Hospital entities and year references
HOSP_RE = re.compile(r"\b((?:[A-Z][a-z]+\s+){1,2})(?:Multispeciality\s+|Multi-speciality\s+|Super\s+Speciality\s+|Speciality\s+|General\s+|Memorial\s+)?"
                     r"(?:Hospitals?|Healthcare|Medical\s+(?:Centre|Center|College)|Health|Institute|Medicity|Clinic|Care)\b")
HOSPITAL_OTHER_RE = re.compile(r"\bsister\b|\bpartner\s+(?:hospital|facility|institution)|\bneighbou?r(?:ing)?\b|\brival\b|\bnearby\s+hospital|"
                      r"\bother\s+hospital|\banother\s+hospital|\bpeer\s+hospital|\bcompetitor|\bshould\s+not\s+be\s+added|\botherhosp[a-z]\b|"
                      r"\bthat\s+hospital\b", re.I)
HOSPITAL_PRIOR_RE = re.compile(r"\b(?:previous|last|prior|preceding|earlier|past)\s+(?:financial\s+|fiscal\s+)?(?:year|period)|\ba\s+year\s+(?:earlier|ago|before)|"
                      r"\bthe\s+year\s+before\b|\byear\s+(?:immediately\s+)?(?:before|preceding)\b|\bcomparison\s+year\b|\bpreceding\s+year\b", re.I)
HOSPITAL_YEAR_RE = re.compile(r"(?<![\d.,₹])\b(?:FY\s?'?(\d{2})\b(?!\s*-)|FY\s?(20\d{2})(?!\s*-\s*\d)|(?:FY\s?)?(20\d{2})\s*-\s*(\d{2})\b|(?:fiscal|year\s+ended\s+March)\s+(20\d{2})|(20[12]\d)(?![.,]\d))", re.I)
EXCL_MONEY = re.compile(r"pharmacy|ancillary|grant|capital|construction|cardiac\s+wing|new\s+wing|operating\s+(?:expenses|costs|surplus)|expenditure|"
                        r"surplus|donation|research|net\s+profit|endowment|loan|debt|salary|salaries|equipment|cafeteria|diagnostic\s+(?:centre|income)|"
                        r"other\s+income|non-patient|investment", re.I)


def _years(s, shift=True):
    out = []
    for m in FY_SHORT_RANGE.finditer(s):
        out.append((m.start(), m.end(), 2000 + int(m[2])))
    for m in HOSPITAL_YEAR_RE.finditer(s):
        if any(a <= m.start() < b for a,b,_ in out):
            continue
        g = m.groups()
        if g[0]:
            y = 2000 + int(g[0])
        elif g[1]:
            y = int(g[1])
        elif g[2]:
            y = int(g[2]) + 1
        elif g[4]:
            y = int(g[4])
        else:
            pre = s[max(0, m.start() - 40):m.start()].lower()
            if re.search(r"(?:founded|established|set\s+up|opened|built|since|registered)\s+(?:in\s+)?$", pre):
                continue
            if re.match(r"^\s*(?:beds|patients|admissions|visits|staff|doctors|crore|lakh|rupees)", s[m.end():m.end() + 14], re.I):
                continue
            y = int(g[5])
        pre2 = s[max(0, m.start() - 30):m.start()]
        if shift and re.search(r"\b(?:before|preceding|prior\s+to|ahead\s+of|earlier\s+than)\s+(?:the\s+)?$", pre2, re.I):
            y -= 1
        elif shift and re.search(r"\b(?:after|following)\s+(?:the\s+)?$", pre2, re.I):
            y += 1
        if 2010 <= y <= 2035:
            out.append((m.start(), m.end(), y))
    return sorted(out)

# Extract inpatient revenue and annualize admissions
FY_SHORT_RANGE = re.compile(r"\bFY\s*['’]?(\d{2})\s*[-–/]\s*(\d{2})\b", re.I)

def year_near(s, start, end):
    """Bind a number to a year in its own clause. A directly attached year wins."""
    dates = _years(s, shift=False)
    ranked = []
    for left, right, year in dates:
        if left < end and right > start:
            continue
        gap = s[end:left] if left >= end else s[right:start]
        if re.search(r"[;.!?]|\b(?:and|but|while|whereas|versus|against|compared|to)\b|\d", gap, re.I):
            continue
        attached = left >= end and re.fullmatch(r"[ ,]*(?:in|for|during|of)?\s*", gap, re.I)
        ranked.append((0 if attached else 1, len(gap), year))
    return min(ranked)[2] if ranked else None


def admissions_comparison(s):
    """Resolve an annual count when a coordinated comparison omits the repeated subject."""
    if not re.search(r"admissions?|admitted|inpatients?", s, re.I) or HOSPITAL_OTHER_RE.search(s):
        return None
    if re.search(r"(?:per|a|each|every) (?:day|month|week)|/(?:day|month|week)|daily|monthly|weekly|originally|first reported", s, re.I):
        return None
    years = _years(s, shift=False)
    if len(set(y for _, _, y in years)) < 2 and not HOSPITAL_PRIOR_RE.search(s):
        return None
    # Preserve commas in thousands. Split on a comparison or a change of clause/subject.
    # Treat both sides of a "from ... to ..." count comparison as separate clauses.
    s = re.sub(r"(\bfrom\s+[\d,]+(?:\s+[^,;.]{1,35})?)\s+to\s+(?=\d)", r"\1 and moved to ", s, flags=re.I)
    parts = re.split(r"\s*(?:;|,?\s+\b(?:and|but|while|whereas|versus|compared (?:with|to)|as against|against)\b)\s*", s, flags=re.I)
    result = []
    for part in parts:
        named = re.search(r"(?:inpatient\s+)?admissions?|admitted|inpatients?", part, re.I)
        for m in NUM_RE.finditer(part):
            if any(a < m.end() and b > m.start() for a,b,_ in _years(part,False)):
                continue
            if any(a <= m.start() < b for a,b,*_ in find_all_money(part)):
                continue
            left = part[:m.start()]
            changed = re.search(r"\b(?:climbed|rose|fell|increased|moved|declined|decreased|grew|dropped|went|edged)\s+(?:(?:up|down)\s+)?to\s*$", left, re.I)
            if re.search(r"outpatient|\bbeds?\b|staff|surger|revenue|procedures?|occupancy", part, re.I):
                continue
            if not named and not changed and not result:
                continue
            year = year_near(part, m.start(), m.end())
            local_dates = {y for _,_,y in _years(part, False)}
            if year is None and len(local_dates) == 1:
                year = next(iter(local_dates))
            if year is None and HOSPITAL_PRIOR_RE.search(part):
                year = -1
            if year is not None and num(m) > 0:
                result.append((year, num(m)))
    if len({y for y,v in result}) > 1:
        latest = max(y for y,v in result)
        values = {v for y,v in result if y == latest}
        if len(values) == 1:
            return latest, values.pop()
    return None


def extract_hospital(text):
    text = normalize_text(pre_normalize(text))
    text = re.sub(r"\(?\s*(?:inpatient\s+(?:plus|and|\+)\s+outpatient|outpatient\s+(?:plus|and|\+)\s+inpatient)(?:\s+care|\s+revenue|\s+services)?\s*\)?", " combined ", text, flags=re.I)
    text = re.sub(r"combining\s+(?:in|out)patient\s+and\s+(?:in|out)patient(?:\s+care|\s+services)?", "combined", text, flags=re.I)
    names = []
    body = '\n'.join(ln for k, ln in enumerate(text.split('\n'))
                     if not re.match(r"^\s*(?:\*\*|#)", ln) and not (k == 0 and len(ln) < 160 and not ln.rstrip().endswith('.')))
    for m in HOSP_RE.finditer(body):
        words = m.group(1).split()
        key = words[-1]
        if key.lower() in ('the', 'our', 'a', 'this', 'each', 'every', 'its', 'which', 'that', 'neighbouring', 'nearby', 'other', 'another', 'per',
                           'total', 'inpatient', 'outpatient', 'patient', 'multispeciality', 'sister', 'partner', 'shri', 'sri', 'sanjeevani', 'city'):
            continue
        typ = 'OTHER' if words[0].lower() in ('sister', 'partner', 'neighbouring', 'rival') else 'NAME'
        if key not in [n for _, n in names]:
            names.append((typ, key))
    main = next((n for t, n in names if t == 'NAME'), None)
    others = [n for t, n in names if n != main]
    for k, w in enumerate(others):
        text = re.sub(rf"\b{re.escape(w)}\b", f'OtherHosp{chr(97 + k)}', text)
    us = report_units(text)
    all_years = [y for _, s, *_ in us for _, _, y in _years(s, shift=False)]
    cur_year = max(all_years) if all_years else None

    qs = []
    sec_other = False
    q_other = False
    last_other = False
    orig_role = None
    last_orig_money = None
    role_hint = None
    money_hint = None

    for kind, s, _, _, _ in us:
        if kind == 'heading':
            sec_other = bool(HOSPITAL_OTHER_RE.search(s)) and not (main and main in s)
            orig_role = None
            continue
        if kind == 'q':
            q_other = bool(HOSPITAL_OTHER_RE.search(s))
            last_other = q_other
            continue
        if kind != 'a':
            q_other = False
        sent_other = sec_other or (kind == 'a' and q_other)
        if kind != 'table' and last_other and re.match(r"^\s*(?:in|for|during|over)\s+[^,.;]{0,45}?,?\s+(?:it|its)\b", s, re.I):
            sent_other = True
        if kind != 'table' and last_other and re.match(r"^\s*(?:its|their|it|they|there|(?:in|for|during)\s+(?:fy\s?'?\d+|fiscal(?:\s+\d+)?|the\s+(?:same\s+)?year)(?:,)?\s+(?:it|its)\b|these\s+are)\b", s, re.I):
            sent_other = True
        if re.search(r"\bmain\s+hospital|\bour\s+(?:own\s+)?hospital|\bthe\s+main\s+(?:facility|institution)", s, re.I) and not sec_other:
            sent_other = False
        if kind == 'table' and not sec_other:
            sent_other = False
        if not sent_other:
            current = admissions_comparison(s)
            if current and (cur_year is None or current[0] == cur_year):
                qs.append(Quantity(role="adm_year", value=current[1], current=True))
        _syl = _years(s)
        _fv = min([mm[0] for mm in find_all_money(s)] + [mm.start() for mm in NUM_RE.finditer(s) if not (2005 <= num(mm) <= 2035)] + [len(s)])
        _sy = [y for p0, _, y in _syl]
        sent_year_prior = bool(_sy) and cur_year is not None and max(_sy) < cur_year and min(p0 for p0, _, _ in _syl) < _fv
        sent_prior = sent_year_prior or bool(re.match(r"^\s*(?:(?:in|during|for|over)\s+)?(?:the\s+)?(?:previous|prior|preceding|last|comparison)\s+(?:financial\s+|fiscal\s+)?year\b|"
                                   r"^[^.;]{0,15}?(?:in\s+|during\s+)?the\s+year\s+(?:immediately\s+)?(?:before|preceding)|"
                                   r"^\s*(?:a|one)\s+year\s+(?:earlier|before)|^\s*(?:by\s+way\s+of\s+|for\s+)comparison", s, re.I))
        for cl in clauses(s):
            other = sent_other
            if HOSPITAL_OTHER_RE.search(cl):
                other = True
                sent_other = True
            if other:
                continue
            money = find_all_money(cl)
            mspans = [(a, b) for a, b, _, _ in money]
            for k, (a, b, v, _) in enumerate(money):
                left = cl[(money[k - 1][1] if k > 0 else 0):a]
                right = cl[b:(money[k + 1][0] if k + 1 < len(money) else len(cl))]
                lab = left[-140:].lower()
                post = right[:80].lower()
                wide = (cl[max(0, a - 170):a]).lower()
                _contra = r"(?:not|never)\s+(?:\w+\s+){0,3}(?:within|in|of|part\s+of|as|into)\s+(?:the\s+)?(?:total\s+)?patient\s+revenue|" \
                          r"(?:different|separate|distinct)\s+(?:\w+\s+){0,3}from\s+(?:the\s+)?(?:total\s+)?patient\s+revenue|rather\s+than\s+patient\s+revenue|" \
                          r"(?:not|never)\s+(?:\w+\s+){0,2}patient\s+revenue|" \
                          r"(?:distinct|separate|different|apart)\s+(?:\w+\s+){0,2}from\s+(?:the\s+)?(?:in|out)patient\s+(?:revenue|services|care|stream)(?![^.;]*\b(?:which|that|whose|where)\b)|" \
                          r"(?:unrelated|unconnected|not\s+related)\s+(?:\w+\s+){0,2}to\s+(?:the\s+)?(?:\w+\s+){0,4}(?:in|out)?patient\s+revenue(?:\s+figures?)?"
                def _strip(t):
                    return re.sub(_contra, lambda mm: mm.group(0) if len(t) - mm.end() < 12 and re.match(r"^\s*(?:of|was|were|at|stood\s+at|:)?\s*$", t[mm.end():]) else " ", t)
                if re.search(r"\b(?:not|never)\s+(?:part of |included in )?(?:the )?(?:total )?patient revenue\s*[:,-]?\s*$",lab):
                    continue
                lab = _strip(lab)
                wide = _strip(wide)
                _pm0 = re.match(r"^\s*(?:was\s+|were\s+|is\s+)?(?:earned\s+|generated\s+|came\s+|derived\s+|contributed\s+|brought\s+in\s+)?(?:of\s+(?:revenue|income)\s+)?(?:from|in|of|for|by)\s+(?:the\s+)?(?:inpatient|outpatient|in-patient|out-patient)", post)
                _ex = max((mm.end() for mm in EXCL_MONEY.finditer(wide)), default=-1)
                _pos = max((mm.end() for mm in re.finditer(r"inpatient|outpatient|patient\s+revenue|combined", wide)), default=-1)
                if not _pm0 and (_ex > _pos or EXCL_MONEY.search(re.split(r"[;,]|\bwhile\b|\bwhereas\b|\band\b", post)[0][:40])):
                    continue
                orig, revised = orig_revised(cl[max(0, a - 120):a], right[:60])
                cues = [('inp_rev', r"inpatient|in-patient|admitted\s+patients|ward\s+revenue|from\s+inpatient"),
                        ('out_rev', r"outpatient|out-patient|opd\b|clinics?\s+revenue"),
                        ('tot_rev', r"total\s+patient\s+revenue|(?<!in)(?<!out)(?<!in-)(?<!out-)patient\s+revenue|total\s+revenue|(?<!in)(?<!out)patient\s+income|"
                                    r"\bcombined\b|overall\s+(?:patient\s+)?revenue")]
                pr = None
                pm = re.match(r"^\s*(?:was\s+|were\s+|is\s+)?(?:earned\s+|generated\s+|came\s+|derived\s+|contributed\s+|brought\s+in\s+)?(?:of\s+(?:revenue|income)\s+)?(?:from|in|of|for|by)\s+(?:the\s+)?(inpatient|outpatient|in-patient|out-patient)", post)
                if pm:
                    pr = 'inp_rev' if pm.group(1).startswith('in') else 'out_rev' if pm.group(1).startswith('out') else 'tot_rev'
                else:
                    best = None
                    for role, pat in cues:
                        ends = [mm.end() for mm in re.finditer(pat, wide)]
                        if role == 'inp_rev':
                            ends = [e for e in ends if not re.search(r"plus\s+outpatient", wide[e:e + 20])]
                        if ends and (best is None or max(ends) > best[0]):
                            best = (max(ends), role)
                    if best is not None:
                        pr = best[1]
                        if pr == 'inp_rev' and re.search(r"(?:share|%|per\s+cent)[^.]{0,40}inpatient", wide[-60:]) and re.search(r"(?<!in)patient\s+revenue", wide):
                            pr = 'tot_rev'
                _cont = False
                if pr is None and money_hint and re.match(r"^\s*(?:it|this|that|the\s+(?:figure|number|amount|total))\b", s, re.I):
                    pr = money_hint
                if pr is None and last_orig_money is not None and (REV_CUE.search((s[:s.find(cl)] if s.find(cl) >= 0 else '')[-80:] + lab[-80:]) or
                                                                   re.search(r"\b(?:replac\w*|supersed\w*)\b|final\s+figure", post[:60])):
                    pr, revised, _cont = last_orig_money, True, True
                if pr is None:
                    continue
                if not _cont and not re.search(r"revenue|income|earn|brought|generated|contributed|billing|receipts|came\s+to|totall?ed|stood|was|were|reached|rose|grew|"
                                 r"climbed|moved|amounted|services", wide[-120:] + ' ' + post[:40]):
                    continue
                seg_l = lab[-60:]
                seg_r = re.split(r"[;,]|\bto\b|\bwhile\b|\bbut\b", post)[0]
                prior = bool(HOSPITAL_PRIOR_RE.search(seg_r[:75])) or (bool(HOSPITAL_PRIOR_RE.search(seg_l)) and not re.search(r"\bto\s*$", seg_l))
                if re.search(r"\bfrom\s*$", lab[-8:]) and re.search(r"\bto\b", post[:80]) and not pm:
                    prior = True
                _is_to = bool(re.search(r"\bto\s*$", lab[-6:]))
                yrs_l = [y for p0, p1, y in _years(cl) if 0 <= a - p1 < 30 and not find_all_money(cl[p1:a])]
                yrs_r = [y for p0, p1, y in _years(cl) if 0 <= p0 - b < 45 and not find_all_money(cl[b:p0])]
                if HOSPITAL_PRIOR_RE.search(seg_l):
                    boundary = re.search(r",\s*(?:which|so|while|whereas)\b",right,re.I)
                    if boundary:
                        yrs_r = [y for p0,p1,y in _years(cl) if b <= p0 < b+min(45,boundary.start())]
                yy = yrs_r or yrs_l
                if yy and cur_year and max(yy) < cur_year and not (_is_to and not yrs_r):
                    prior = True
                if yy and cur_year and max(yy) == cur_year:
                    prior = False
                prior = prior or (sent_prior and not (yy and cur_year and max(yy) == cur_year))
                if _is_to and not HOSPITAL_PRIOR_RE.search(post[:50]) and not (yrs_r and cur_year and max(yrs_r) < cur_year):
                    prior = False
                if re.search(r"\b(?:previous|prior|preceding) (?:fiscal |financial )?year\s+(?:came|amounted) to\s*$",lab,re.I):
                    prior = True
                if orig:
                    last_orig_money = pr
                qs.append(Quantity(role=pr, value=v, orig=orig, revised=revised, prior=prior))

            for m in re.finditer(r"(\d+(?:\.\d+)?)\s*%", cl):
                if any(a0 <= m.start() < b0 for a0, b0 in mspans):
                    continue
                pre = cl[max(0, m.start() - 100):m.start()].lower()
                post = cl[m.end():m.end() + 90].lower()
                ctx = pre[-100:] + ' ' + post[:70]
                if re.search(r"occupanc|bed|growth|grew|rose|increase|fell|decline|margin|icu|mortality|utilisation|utilization", pre[-40:] + post[:25]):
                    continue
                if not re.search(r"revenue|income|billing|\btotal\b|\bshare\b|contribut|accounted|of\s+which", ctx):
                    continue
                ip = [mm for mm in re.finditer(r"inpatient|in-patient", ctx)]
                op = [mm for mm in re.finditer(r"outpatient|out-patient", ctx)]
                c0 = len(pre[-100:])
                ib = [c0 - mm.start() for mm in ip if mm.start() < c0]
                ob = [c0 - mm.start() for mm in op if mm.start() < c0]
                if ib or ob:
                    role = 'inp_share' if (min(ib) if ib else 999) <= (min(ob) if ob else 999) else 'out_share'
                else:
                    di = min((abs(mm.start() - c0) for mm in ip), default=999)
                    do = min((abs(mm.start() - c0) for mm in op), default=999)
                    if di == 999 and do == 999:
                        continue
                    role = 'inp_share' if di <= do else 'out_share'
                prior = bool(HOSPITAL_PRIOR_RE.search(pre[-50:])) or sent_prior
                qs.append(Quantity(role=role, value=float(m.group(1)) / 100, prior=prior, orig=False, revised=False))

            for m in NUM_RE.finditer(cl):
                if any(a < m.end() and b > m.start() for a,b,_ in _years(cl,False)):
                    continue
                if any(a0 <= m.start() < b0 for a0, b0 in mspans):
                    continue
                v = num(m)
                if 2005 <= v <= 2035 and ',' not in m.group(0):
                    continue
                pre = cl[max(0, m.start() - 110):m.start()].lower()
                post = cl[m.end():m.end() + 80].lower()
                if re.match(r"^\s*%", post):
                    continue
                if re.match(r"^\s*(?:beds?|doctors?|nurses?|staff|surgeons?|surger|surgical|procedures?|operations?|outpatient|visits?|consultations?|icu|cities|departments?|"
                            r"specialit|years?|crore|lakh|million|billion|km|acres?)\b", post):
                    continue
                after = re.split(r"[;]|\bwhile\b|\bwhereas\b", post)[0]
                lab = re.sub(r"\s+", " ", pre)[-110:]
                if re.match(r"^\s*(?:and|but|while|whereas)\b", cl, re.I) and s.find(cl) > 0:
                    lab = re.sub(r"\s+", " ", s[:s.find(cl)] + cl[:m.start()])[-110:].lower()
                lab_ny = HOSPITAL_YEAR_RE.sub(" ", lab)
                is_adm = bool(re.match(r"^\s*(?:inpatient\s+|in-patient\s+|patient\s+|new\s+)?(?:admissions?|inpatients?|patients\s+(?:were\s+)?admitted|in-patients)\b", post)) or \
                    bool(re.search(r"(?:admission|admitted|inpatients?\s+(?:treated|admitted)|treated\s*$)[^.;]{0,105}$", lab_ny)) and \
                    not re.search(r"(?:outpatient|visit|consultation|surger|surgic|caseload|procedure|bed|doctor|staff)[^.;]{0,25}$", lab[-30:])
                if not is_adm and role_hint and re.match(r"^\s*(?:it|this|that|the\s+(?:figure|number|count|total))\b", s, re.I) and \
                        not re.search(r"revenue|crore|lakh|billion|million", s[:m.start() + 1], re.I):
                    is_adm = True
                    _forced = role_hint
                else:
                    _forced = None
                if not is_adm:
                    if orig_role is not None and orig_role.startswith('adm') and (REVISED_TO.search(pre[-60:]) or
                                                                                  re.match(r"^\s*(?:\w+\s+){0,2}(?:replaces|supersedes)\b", post)):
                        role = orig_role
                    else:
                        continue
                elif _forced:
                    role = _forced
                else:
                    ctxp = lab_ny[-110:] + ' ' + after[:60]
                    role = 'adm_month' if re.search(r"per\s+month|a\s+month|each\s+month|every\s+month|monthly|/\s*month|month\s+on\s+average", ctxp) else \
                        'adm_day' if re.search(r"per\s+day|a\s+day|each\s+day|every\s+day|daily|/\s*day", ctxp) else 'adm_year'
                orig, revised = orig_revised(pre[-120:], post[:60])
                if orig:
                    orig_role = role
                elif orig_role == role and REV_CUE.search(pre[-70:]):
                    revised = True
                seg_r = re.split(r"[;,]|\bto\b|\bwhile\b|\bbut\b", post)[0]
                prior = bool(HOSPITAL_PRIOR_RE.search(seg_r[:70])) or (bool(HOSPITAL_PRIOR_RE.search(lab[-60:])) and not re.search(r"\bto\s*$", lab))
                if re.match(r"^\s*(?:compared\s+(?:with|to)|as\s+against|against|versus|up\s+from|down\s+from)\b", cl, re.I):
                    prior = True
                if re.search(r"\bfrom\s*$", lab[-8:]) and re.search(r"\bto\b", post[:60]):
                    prior = True
                _is_to_c = bool(re.search(r"\bto\s*$", pre[-6:]))
                yrs_l = [y for p0, p1, y in _years(cl) if 0 <= m.start() - p1 < 40 and not _is_to_c]
                yrs_r = [y for p0, p1, y in _years(cl) if 0 <= p0 - m.end() < 40 and not [mm for mm in NUM_RE.finditer(cl[m.end():p0]) if not (2005 <= num(mm) <= 2035)]]
                if HOSPITAL_PRIOR_RE.search(pre) and re.match(r"^\s*(?:inpatient\s+)?admissions?\s*,\s*(?:so|which|although|whereas|while)\b",post,re.I):
                    # A new clause may mention the reporting year after a prior-year count.
                    boundary = re.search(r",\s*(?:so|which|although|whereas|while)\b",post,re.I).start()
                    yrs_r = [year for a,b,year in _years(cl) if m.end() <= a < m.end()+boundary]
                yy = yrs_r or yrs_l
                if yy and cur_year and max(yy) < cur_year:
                    prior = True
                if yy and cur_year and max(yy) == cur_year:
                    prior = False
                prior = prior or (sent_prior and not (yy and cur_year and max(yy) == cur_year))
                qs.append(Quantity(role=role, value=v, orig=orig, revised=revised, prior=prior))

        last_other = sent_other
        if not re.search(r"\d", HOSPITAL_YEAR_RE.sub(" ", s)):
            money_hint = ('inp_rev' if re.search(r"inpatient\s+(?:services\s+)?revenue|revenue\s+from\s+inpatient|inpatient\s+services?\s+(?:earned|brought|generated)", s, re.I) else
                          'out_rev' if re.search(r"outpatient\s+(?:services\s+)?revenue|revenue\s+from\s+outpatient", s, re.I) else
                          'tot_rev' if re.search(r"total\s+patient\s+revenue|(?<!in)(?<!out)\bpatient\s+revenue", s, re.I) else None)
        else:
            money_hint = None
        if not re.search(r"\d", HOSPITAL_YEAR_RE.sub(" ", s)) and re.search(r"admission|admitted", s, re.I):
            role_hint = 'adm_month' if re.search(r"per\s+month|monthly", s, re.I) else 'adm_day' if re.search(r"per\s+day|daily", s, re.I) else 'adm_year'
        else:
            role_hint = None
    return qs

def parse_hospital(text):
    qs = extract_hospital(text)
    inp = pick_current_period(qs, 'inp_rev')
    if inp is None:
        tot = pick_current_period(qs, 'tot_rev')
        if tot is not None:
            sh, osh, out = pick_current_period(qs, 'inp_share'), pick_current_period(qs, 'out_share'), pick_current_period(qs, 'out_rev')
            if sh is not None:
                inp = tot * sh
            elif out is not None:
                inp = tot - out
            elif osh is not None:
                inp = tot * (1 - osh)
    dated = [q.value for q in qs if q.role == 'adm_year' and q.current and not q.orig and not q.prior]
    adm = dated[-1] if dated and not any(q.revised and q.role == 'adm_year' and not q.prior and not q.orig for q in qs) else pick_current_period(qs, 'adm_year')
    if adm is None:
        mth = pick_current_period(qs, 'adm_month')
        if mth is not None:
            adm = mth * 12
        else:
            day = pick_current_period(qs, 'adm_day')
            if day is not None:
                adm = day * 365
    x = np.array([np.nan if inp is None else inp, np.nan if adm is None else adm], float)
    return x

HOSPITAL_LABELS = {
    "revenue": r"inpatient (?:services?|care)?\s*(?:revenue|income)|revenue from inpatient|inpatient services|inpatient care|brought in|earned",
    "total": r"total patient revenue|patient revenue total",
    "outpatient": r"outpatient (?:services?)?\s*(?:revenue|income)|revenue from outpatient|outpatient services",
    "admissions": r"inpatient admissions|admissions|admitted",
    "share": r"inpatient|admitted patient",
}

def recover_hospital(text, inputs):
    """Fill missing fields using label distance, table position and correction cues."""
    scored = defaultdict(list)
    for line in normalize_text(pre_normalize(text)).splitlines():
        if re.search(r"sister|neighbour|rival|partner hospital", line, re.I):
            continue
        table = line.strip().startswith("|")
        sentences = [" | ".join(v.strip() for v in line.strip().strip("|").split("|"))] if table else split_sentences(line)
        for sentence in sentences:
            lower = sentence.lower()
            comparison = admissions_comparison(sentence)
            if comparison:
                scored["admissions"].append((comparison[0], float("inf"), comparison[1]))
            money = find_all_money(sentence)
            quantities = [(a,b,v,True) for a,b,v,_ in money]
            for m in re.finditer(r"\d[\d,]*(?:\.\d+)?", sentence):
                if not any(a <= m.start() < b for a,b,_,_ in money):
                    quantities.append((m.start(), m.end(), float(m[0].replace(",","")), False))
            for a,b,v,is_money in quantities:
                local = lower[max(0,a-82):a+85]
                context = lower[max(0,a-65):a] + " " + lower[b:b+65]
                adjust = 10 * table - 9 * bool(re.search(r"prior|previous|last year|year before|year earlier|initial|old figure",context))
                adjust += 4 * bool(re.search(r"revised|restated|corrected|updated|replaces|final|current",context))
                adjust -= 10 * bool(re.search(r"first reported|original|initially",context))
                if table and " | " not in sentence[:a]:
                    adjust -= 8
                for role, pattern in HOSPITAL_LABELS.items():
                    if is_money != (role in ("revenue","total","outpatient")):
                        continue
                    matches = list(re.finditer(pattern,lower))
                    if not matches:
                        continue
                    dist = min(max(m.start()-b, a-m.end(), 0) for m in matches)
                    score = adjust - min(dist,180)*.15
                    val = v
                    if role == "revenue":
                        if "inpatient" not in lower: continue
                        score -= 24 * ("outpatient" in local and "inpatient" not in local)
                        score -= 15 * bool(re.search("patient revenue|total revenue",local))
                    if role == "share":
                        if "%" not in lower or not 10 <= v <= 99: continue
                        val /= 100
                    if role == "admissions":
                        score -= 30 * (v < 400 or v > 250000)
                        score -= 45 * any(a0 <= a < b0 for a0,b0,_ in _years(sentence,shift=False))
                        score -= 16 * ("outpatient" in local)
                        score -= 10 * bool(re.search("previous|prior|year earlier|last year|year before",local))
                        for frequency, periods in ((r"per day|daily|each day",365), (r"per month|monthly|each month",12), (r"per week|weekly|each week",52)):
                            if re.search(frequency,lower):
                                val *= periods
                                score += 3
                                break
                        if not 400 <= val <= 200000: continue
                    if role == "admissions" and any(l < b and r > a for l,r,_ in _years(sentence,False)):
                        continue
                    year = year_near(sentence, a, b)
                    scored[role].append((year or 0, score, val))

    def best(role):
        return max(scored[role], key=lambda item:item[:2])[2] if scored[role] else np.nan
    revenue = best("revenue")
    if not np.isfinite(revenue):
        total, share = best("total"),best("share")
        revenue = total*share if np.isfinite(share) else total-best("outpatient")
    return np.where(np.isfinite(inputs),inputs,[revenue,best("admissions")])


## Farmer reports
Extract yield per acre, realised sale price per quintal, and total cultivation cost per acre. Benchmark prices and individual cost components have separate roles.


In [5]:
# Farmer cooperatives, seasons and benchmark prices
ORG_RE = re.compile(r"\b([A-Z][a-z]+)\s+(?:Agro\s+Producer\s+Cooperative|Agro\s+Cooperative|Farmers?\s+Producer\s+Company|Agro\s+FPO|Krishak\s+Sahakari\s+Samiti|Kisan\s+(?:FPO|Samiti|Sangh|Producer\s+Company)|"
                    r"FPO|Farmer\s+Producer\s+(?:Organisation|Organization|Company)|Producer\s+Company|Cooperative|Samiti|Sangh)\b")
FARMER_OTHER_RE = re.compile(r"\bneighbou?r(?:ing|s)?\b|\brival\b|\bthat\s+(?:cooperative|fpo|samiti|sangh|society|company|organisation)\b|\bthe\s+other\s+(?:cooperative|fpo|samiti)\b|\bnearby\s+(?:cooperative|fpo|samiti)|\bother\s+(?:cooperative|fpo|samiti)|"
                      r"\bpeer\b|\bcompetitor|\botherorg[a-z]\b", re.I)
FARMER_PRIOR_RE = re.compile(r"\b(?:previous|last|prior|preceding|earlier|past)\s+(?:season|year|harvest|crop)|\b(?:a|one|the)\s+(?:year|season)\s+(?:earlier|ago|before)|"
                      r"\bthe\s+season\s+before\b|\byear\s+before\b", re.I)
MSP_RE = re.compile(r"\bMSP\b|minimum\s+support\s+price|support\s+price|mandi|modal|market\s+(?:price|rate|benchmark)|open[\s-]market|"
                    r"policy\s+price|benchmark|government\s+price|procurement\s+price|district\s+average|state\s+average", re.I)


FARMER_YEAR_RE = re.compile(r"(?<![\d.,₹])\b(20[12]\d)(?![.,]\d)(?:\s*-\s*(\d{2}))?\b")

def _years_in(s):
    out = []
    for m in FARMER_YEAR_RE.finditer(s):
        pre = s[max(0, m.start() - 40):m.start()].lower()
        if re.search(r"(?:registered|founded|established|formed|set\s+up|incorporated|since)\s+(?:in\s+)?$", pre):
            continue
        if re.match(r"^\s*(?:mm|millimetres|tonnes|quintals|acres|members|farmers|rupees|crore|lakh)", s[m.end():m.end() + 14], re.I):
            continue
        y = int(m.group(1))
        out.append((m.start(), m.end(), y))
    return out

# Extract yield, realised price and cultivation cost per acre
def extract_farmer(text):
    text = normalize_text(pre_normalize(text))
    names = []
    for m in ORG_RE.finditer(text):
        if m.group(1) not in names and m.group(1).lower() not in ('the', 'our', 'a', 'this', 'each', 'every', 'agro', 'kisan', 'farmers', 'farmer',
                                                                    'producer', 'krishak', 'sahakari', 'neighbouring', 'neighboring', 'rival', 'other', 'nearby'):
            names.append(m.group(1))
    main_word = names[0] if names else None
    other_words = [w for w in names[1:] if w != main_word]
    for k, w in enumerate(other_words):
        text = re.sub(rf"\b{re.escape(w)}\b", f'OtherOrg{chr(97 + k)}', text)
    if main_word:
        text = ORG_RE.sub(lambda mm: 'MainOrg' if mm.group(1) == main_word else mm.group(0), text)
        main_word = 'MainOrg'
    us = report_units(text)
    all_years = [y for _, s, *_ in us for _, _, y in _years_in(s)]
    cur_year = max(all_years) if all_years else None

    qs = []
    sec_other = False
    q_other = False
    last_other = False
    orig_role = None
    last_orig_money = None

    for kind, s, _, _, _ in us:
        if kind == 'heading':
            sec_other = bool(FARMER_OTHER_RE.search(s)) and not (main_word and main_word in s)
            orig_role = None
            continue
        if kind == 'q':
            q_other = bool(FARMER_OTHER_RE.search(s))
            last_other = q_other
            continue
        if kind != 'a':
            q_other = False
        sent_other = sec_other or (kind == 'a' and q_other)
        if kind != 'table' and last_other and re.match(r"^\s*(?:its|their|it|they|there)\b", s, re.I):
            sent_other = True
        if kind == 'table' and not sec_other:
            sent_other = False
        sent_prior = bool(re.match(r"^[^.;]{0,25}?(?:in\s+the\s+|during\s+the\s+|for\s+the\s+)?(?:last|previous|prior|preceding)\s+(?:season|year|harvest)|"
                                   r"^\s*(?:compared\s+(?:with|to)\s+)?(?:a|one|the)\s+(?:season|year)\s+(?:earlier|before)", s, re.I)) and \
            not re.search(r"this\s+(?:season|year)|current\s+season", s[:(re.search(r"\d", s).start() if re.search(r"\d", s) else len(s))], re.I)
        for cl in clauses(s):
            other = sent_other
            if FARMER_OTHER_RE.search(cl):
                other = True
                sent_other = True
            if other:
                continue
            money = find_all_money(cl)
            mspans = [(a, b) for a, b, _, _ in money]
            for k, (a, b, v, _) in enumerate(money):
                left = cl[(money[k - 1][1] if k > 0 else 0):a]
                right = cl[b:(money[k + 1][0] if k + 1 < len(money) else len(cl))]
                post = right[:60].lower()
                lab = (left[-120:]).lower()
                wide = cl[max(0, a - 160):a].lower()
                if re.search(r"subsid|grant|\bloans?\b|credit\s+(?:limit|line|facility)|crop\s+loan|insurance|warehouse|storage|capacity|turnover|revenue\s+of|total\s+sales|sales\s+(?:revenue|value)|dividend|share\s+capital|"
                             r"fertili[sz]er\s+alone|of\s+which|seed\s+alone|labour\s+alone|irrigation\s+alone|equipment|fertili[sz]er\s+cost|"
                             r"one\s+component|component\s+of|fertili[sz]er\s+(?:spend|expense|bill)", lab[-70:] + ' ' + re.split(r"[,;]", post)[0][:40]) \
                        and not re.search(r"cultivation\s+cost|cost\s+of\s+cultivation", lab[-40:]):
                    continue
                _lft = (s[:s.find(cl)] + cl[:a]).lower() if s.find(cl) >= 0 else left.lower()
                if re.search(r"fertili[sz]er", _lft) and re.search(r"\balone\b|component|\bonly\b|by\s+itself|\bitself\b|of\s+which", _lft) and \
                        not re.search(r"(?:this|the)\s+(?:per[\s-]acre\s+)?total\s*[:=]?\s*(?:is|was|of|stood\s+at)?\s*$", left.lower()):
                    continue
                orig, revised = orig_revised(cl[max(0, a - 110):a], right[:60])
                _pa_post = re.match(r"^\s*(?:per|a|/|for\s+each|each)\s*acre", post)
                per_q = re.match(r"^\s*(?:per|a|/)\s*(?:quintal|qtl|q\b)", post) or (not _pa_post and (
                    re.search(r"per\s+quintal|/\s*quintal", left.lower()) or re.search(r"\bprice\b|realis|fetched|sold\s+at", lab[-90:])) and
                    not re.search(r"cost|spent|spend|expense|subsid", lab[-50:]))
                per_acre = re.match(r"^\s*(?:per|a|/|for\s+each|each)\s*acre", post) or (re.search(r"per\s+acre|/\s*acre", wide[-150:]) and not per_q and
                                                                                                    not re.search(r"all\s+members|together|in\s+total|combined", lab[-80:]))
                lab_nc = re.sub(r"(?:different|separate|distinct|apart)\s+(?:\w+\s+){0,3}from\s+[^,.;]*?realis(?:ed|ation|e|ing)?\b(?!\s+(?:\w+\s+){0,2}(?:of|was|at|is|:)\s*$)|"
                                r"not\s+(?:what|the\s+price|the\s+one)\s+[^,.;]*?realis\w*|"
                                r"rather\s+than\s+[^,.;]*?realis\w*", " ", lab)
                ctx = lab_nc[-90:] + ' ' + post[:50]
                if per_q:
                    _msp = max((mm.end() for mm in MSP_RE.finditer(lab_nc[-90:])), default=-1)
                    _rl = max((mm.end() for mm in re.finditer(r"realis|obtained|fetched|sold\s+at|received|got|achieved|earned|selling\s+price", lab_nc[-90:])), default=-1)
                    _post_real = re.match(r"^\s*(?:per\s+quintal\s*)?(?:as|being)\s+(?:the|its|our)\s+(?:average\s+)?(?:realised|realized|actual|achieved|obtained)", post)
                    if (_msp > _rl or (_msp >= 0 and _rl < 0)) and not _post_real:
                        continue
                    if MSP_RE.search(re.split(r"[;,]", post)[0][:40]):
                        continue
                    if not re.search(r"realis|obtained|fetched|sold|sales|received|got|average\s+price|price\s+(?:per|realised)|earned|achieved|"
                                     r"realisation|price\s+of|selling", ctx) and MSP_RE.search(ctx):
                        continue
                    role = 'price'
                elif per_acre:
                    if not re.search(r"cost|spent|spend|expense|outlay|cultivat|seed|labour|input", wide[-150:] + ' ' + post[:50]) and last_orig_money != 'cost_acre':
                        continue
                    role = 'cost_acre'
                elif last_orig_money is not None and (REV_CUE.search(lab[-80:]) or re.search(r"\b(?:replaces|supersedes)\b", post[:40])):
                    role = last_orig_money
                    revised = True
                elif re.search(r"spent\s+(?:a\s+)?(?:combined|total|together|in\s+all)|combined\s+(?:spend|cost|outlay)|collectively\s+spent", lab[-80:]) or \
                        (re.search(r"\bspent\b|\bspend\b", lab[-60:]) and re.match(r"^\s*(?:in\s+total\s+)?on\s+(?:seed|inputs|cultivation)", post)):
                    role = 'cost_total'
                elif re.search(r"cultivation\s+cost|cost\s+of\s+cultivation|cultivation\s+(?:expenses|spend)|spent\s+on\s+cultivation|cost[^.;]{0,40}all\s+members|"
                               r"total\s+cost|cultivation[^.;]{0,30}(?:came|totall?ed|was|amounted)", wide[-150:]) or \
                        re.search(r"all\s+members|together|in\s+total|total|combined", ctx) and re.search(r"cost|spent|spend|covering\s+seed|seed,\s+fertili", wide[-150:] + ' ' + post[:60]):
                    role = 'cost_total'
                else:
                    continue
                prior = bool(FARMER_PRIOR_RE.search(lab[-70:] + ' ' + post[:40]))
                yrs = [y for p0, p1, y in _years_in(cl) if abs(p0 - a) < 60]
                if yrs and cur_year and max(yrs) < cur_year and not re.search(r"\bto\b", cl[a:a + 30]):
                    prior = True
                if orig:
                    last_orig_money = role
                qs.append(Quantity(role=role, value=v, orig=orig, revised=revised, prior=prior or sent_prior))

            for m in NUM_RE.finditer(cl):
                if any(a0 <= m.start() < b0 for a0, b0 in mspans):
                    continue
                v = num(m)
                post = cl[m.end():m.end() + 70].lower()
                pre = cl[max(0, m.start() - 100):m.start()].lower()
                if 2005 <= v <= 2035 and ',' not in m.group(0) and not re.match(r"^\s*(?:acres?|quintals?|tonnes?)", post):
                    continue
                role = None
                unit_mult = 1.0
                if re.match(r"^\s*(?:member|members|farmers?|villages?|mm|millimetres?|cm|years?|per\s*cent|%|crops?|districts?|blocks?|staff|employees|"
                            r"women|households?|families|hectares?|km|metres?|days?|months?)\b", post):
                    continue
                if re.match(r"^\s*(?:quintals?|qtl)\s*(?:per|/|an?|each)\s*acre", post):
                    role = 'yield'
                elif re.match(r"^\s*(?:kg|kilograms?|kilos?)\s*(?:per|/|an?|each)\s*acre", post):
                    role, unit_mult = 'yield', 0.01
                elif re.match(r"^\s*(?:tonnes?|tons?)\s*(?:per|/|an?|each)\s*acre", post):
                    role, unit_mult = 'yield', 10.0
                elif re.match(r"^\s*(?:quintals?|qtl)\b", post):
                    if re.search(r"capacity|storage|warehouse|procure|sold\s+to|stock", pre[-50:] + post[:30]):
                        continue
                    role = 'production'
                elif re.match(r"^\s*(?:tonnes?|tons?|t\b)", post):
                    if re.search(r"capacity|storage|warehouse|procure|stock|godown", pre[-50:] + post[:40]):
                        continue
                    role, unit_mult = 'production', 10.0
                elif re.match(r"^\s*acres?\b", post):
                    if re.search(r"per\s+acre", post[:12]):
                        continue
                    role = 'area'
                else:
                    labt = re.sub(r"\s+", " ", pre[-80:])
                    if re.search(r"yield[^.;:]{0,60}(?:\(quintals?\s+per\s+acre\)|quintals?\s+per\s+acre)?[^.;]{0,25}[:]?\s*$|quintals?\s+per\s+acre\)?\s*:?\s*$", labt) and \
                            re.search(r"yield", labt[-70:]):
                        role = 'yield'
                    elif re.search(r"(?:area|acreage|acres)[^.;]{0,50}:?\s*$", labt) and re.search(r"area|acreage|acres", labt[-60:]) and \
                            not re.search(r"warehouse|storage|capacity", labt[-60:]):
                        role = 'area'
                    elif re.search(r"\bproduction\b|\boutput\b|\bharvest(?:ed)?\b|\bproduce\b", labt[-60:]) and not re.search(r"capacity|storage|warehouse|producer", labt[-60:]):
                        role = 'production'
                        if re.search(r"\btonnes?\b|\btons?\b", labt[-40:]):
                            unit_mult = 10.0
                    elif orig_role is not None and REVISED_TO.search(pre[-60:]):
                        role = orig_role
                    else:
                        continue
                if role is None:
                    continue
                orig, revised = orig_revised(pre[-110:], post[:60])
                if orig:
                    orig_role = role
                elif orig_role == role and REV_CUE.search(pre[-70:]):
                    revised = True
                def _notyear(mm):
                    vv = num(mm)
                    return not (2005 <= vv <= 2035 and ',' not in mm.group(0))
                _prev_end = max([mm.end() for mm in NUM_RE.finditer(cl[:m.start()]) if _notyear(mm)] + [mm[1] for mm in mspans if mm[1] <= m.start()] + [0])
                _nxt = [mm.start() for mm in NUM_RE.finditer(cl[m.end():]) if _notyear(mm)]
                left_seg = cl[_prev_end:m.start()].lower()
                right_seg = cl[m.end():m.end() + (_nxt[0] if _nxt else 80)].lower()
                right_seg = re.split(r"[;,]|\bto\b|\bwhile\b|\bwhereas\b|\bbut\b", right_seg)[0]
                prior = bool(FARMER_PRIOR_RE.search(right_seg[:50])) or (bool(FARMER_PRIOR_RE.search(left_seg[-70:])) and not re.search(r"\bto\s*$", left_seg) and
                                                                  not re.search(r"this\s+(?:season|year)|current\s+season", right_seg[:40]))
                if (not prior) and not left_seg.strip() and FARMER_PRIOR_RE.search(pre[-60:]) and not re.search(r"\bto\s*$", pre[-10:]):
                    prior = True
                if re.search(r"this\s+(?:season|year)|current\s+season|latest\s+season", right_seg[:40]):
                    prior = False
                if re.search(r"\bfrom\s*$", pre[-8:]) and re.search(r"\bto\s+\d", post[:60]):
                    prior = True
                _ls0 = m.start() - len(left_seg)
                yrs = [y for p0, p1, y in _years_in(cl) if (_ls0 <= p0 and p1 <= m.start()) or (m.end() <= p0 and p1 <= m.end() + len(right_seg) + 25 and
                                                                                         not NUM_RE.search(cl[m.end():p0].replace(str(y), '')))]
                if yrs and cur_year and max(yrs) < cur_year:
                    prior = True
                if re.search(r"not\s+stated|not\s+disclosed|not\s+given", post[:40]):
                    continue
                _sp = sent_prior and not re.search(r"this\s+(?:season|year)|current\s+season", left_seg + ' ' + right_seg)
                qs.append(Quantity(role=role, value=v * unit_mult, orig=orig, revised=revised, prior=prior or _sp))

        last_other = sent_other
    return qs

def parse_farmer(text):
    qs = extract_farmer(text)
    area = pick_current_period(qs, 'area')
    yld = pick_current_period(qs, 'yield')
    if yld is None:
        prod = pick_current_period(qs, 'production')
        if prod is not None and area:
            yld = prod / area
    for q in qs:   # scale sanity: per-acre costs are thousands of rupees, member totals are lakhs/crores
        if q.role == 'cost_total' and q.value < 2.5e5:
            q.role = 'cost_acre'
        elif q.role == 'cost_acre' and q.value > 1.5e6:
            q.role = 'cost_total'
    price = pick_current_period(qs, 'price')
    cost = pick_current_period(qs, 'cost_acre')
    if cost is None:
        tot = pick_current_period(qs, 'cost_total')
        if tot is not None and area:
            cost = tot / area
    x = np.array([np.nan if yld is None else yld, np.nan if price is None else price,
                  np.nan if cost is None else cost], float)
    return x


## Real Estate reports
Extract total apartments, booked apartments, and current price per apartment. A complete floor plan provides an inventory check, and an explicit revision takes priority over the original total.


In [6]:
# Property entities and apartment count roles
PROPERTY_OTHER_RE = re.compile(r"\b(?:second|2nd|another|other)\s+phase\b|\bphase[\s-]?(?:2|ii|two)\b|"
                      r"\bsister\s+project|\bprevious\s+project|\bearlier\s+project|\bneighbou?r(?:ing)?\b|\brival\b|"
                      r"\bcompetitor|\bnearby\s+project|\bother\s+(?:project|development)|\banother\s+(?:project|development)|"
                      r"\bsold\s+out\b|\botherproject[a-z]\b", re.I)
LAUNCH_RE = re.compile(r"(?<!not\s)(?<!since\s)\blaunch|\bat\s+the\s+start|\bopened\s+(?:for\s+)?(?:sale|bookings?)|\binitial(?:ly)?\s+(?:price|rate)|"
                       r"earlier\s+and\s+lower|\bwhen\s+(?:the\s+project|it|bookings?|sales|booking)\s+(?:first\s+)?(?:opened|was\s+launched|were\s+launched|began|started|opened\s+up)|\bbookings?\s+opened|\bearly\s+(?:buyers|entrants|investors|purchasers)|"
                       r"\bfirst\s+offered|\boriginal\s+(?:price|rate)|\bbefore\s+(?:the\s+)?(?:increase|rise|hike)|\bstarting\s+(?:price|rate)|"
                       r"\bopening\s+(?:price|rate)|\bat\s+opening|\bat\s+the\s+outset|\bin\s+the\s+beginning|"
                       r"\bwhen\s+sales\s+(?:began|started|opened)", re.I)
CURRENT_RE = re.compile(r"(?<!not\s)(?<!not\sthe\s)\bcurrent|\bnow\b|\btoday|\bat\s+present|\bpresent(?:ly)?\b|\bprevailing|\bin\s+force|\blatest|\bthese\s+days", re.I)
PROJ_RE = re.compile(r"\b([A-Z][a-z]+)\s+(?:Residency|Skyline|Gardens|Enclave|Greens|Heights|Towers)\b")
SPEND_RE = re.compile(r"spent|spend|construction|land\s+cost|capex|capital|marketing|loan|debt|revenue|sales\s+value|collections?|"
                      r"receipts?|investment|crore\s+on|fund|put\s+(?:in|into)|invested|outlay|budget", re.I)


AUX = r"(?:(?:are|is|have|has|had|were|was|been|already|so\s+far|now|of\s+these|of\s+them|in\s+all|in\s+total|to\s+date|remain|still|yet|as|recorded\s+as|listed\s+as|counted\s+as|reported\s+as)\s+){0,4}"

def role_count(cl, m):
    pre = cl[max(0, m.start() - 90):m.start()].lower()
    post = cl[m.end():m.end() + 70].lower()
    if re.match(r"^\s*(?:residential\s+)?towers?\b", post):
        return 'towers'
    if re.match(r"^\s*(?:residential\s+)?(?:floors?|storeys?|stories|levels)\b", post):
        return 'floors'
    if re.match(r"^\s*(?:apartments?|units?|homes|flats)\s+(?:on\s+(?:every|each)\s+floor|per\s+floor|a\s+floor|on\s+each\s+level)", post):
        return 'per_floor'
    if re.match(r"^\s*(?:sq(?:uare)?\.?\s*(?:ft|feet|foot)|sq\s*ft)", post):
        if re.search(r"club|amenit|land|plot|retail|commercial|podium|garden|park|office", pre[-50:] + post[:30]):
            return None
        return 'size'
    if re.match(r"^\s*(?:apartments?|units?|homes|flats)?\s*" + AUX + r"(?:unsold|unbooked|available|remaining|remain\s+(?:unsold|available)|"
                r"yet\s+to\s+be\s+(?:sold|booked)|left\s+to\s+sell|on\s+offer)", post) or \
            re.match(r"^\s*(?:apartments?|units?|homes|flats)\b[^.;,%\d]{0,45}?\b(?:remain(?:ing)?\s+|are\s+(?:still\s+)?|still\s+)(?:unsold|unbooked|available|yet\s+to\s+be\s+(?:sold|booked))", post):
        return 'unsold'
    if (re.match(r"^\s*(?:apartments?|units?|homes|flats)?\s*" + AUX + r"(?:booked|sold|reserved|taken)", post) or
            re.match(r"^\s*(?:apartments?|units?|homes|flats)\b[^.;,%\d]{0,45}?\b(?:are|have\s+been|were|had\s+been)\s+(?:already\s+|so\s+far\s+)?(?:booked|sold|reserved)\b", post) or
            re.match(r"^\s*(?:apartments?|units?|homes|flats)\b(?!\s+in\s+total)[^.;%\d]{0,60}?\b(?:apartments?\s+)?(?:booked|bookings)\b", post)) and \
            not re.match(r"^\s*(?:apartments?|units?)?\s*sold\s+out", post):
        return 'booked'
    if re.match(r"^\s*(?:acres?|hectares?|km|kilomet|metres?|meters?|years?|months?|%)", post):
        return None
    if re.match(r"^\s*(?:apartments?|units?|homes|flats)\s+in\s+(?:the\s+(?:project|development|scheme)|total|all)\b", post) and \
            not re.search(r"\b(?:booked|sold|unsold|unbooked)\b", post[:45]):
        return 'total'
    if re.match(r"^\s*[a-z]", post) and not re.match(
            r"^\s*(?:apartments?|units?|homes|flats|towers?|floors?|residential|storeys?|stories|levels?|sq|square|are|have|has|had|were|was|is|"
            r"of|in|out|still|already|remain\w*|and|as|so|to|with|now|left|across|booked|sold|unsold|unbooked|replaces|supersedes|which|that|"
            r"for|on|per|each|every|at|by|from|than|available|reserved|after|since|according|or|but|while|whereas)\b", post):
        return None
    if re.match(r"^\s*(?:apartments?|units?|homes|flats)\b(?!\s+(?:on|per)\s+(?:every|each|a|the)?\s*(?:floor|level))", post):
        if re.search(r"\b(?:booked|sold|reserved)\s*(?:stand\s+at|at|reached|of|:|to)?\s*$|\bbookings?\s+(?:have\s+)?(?:reached|stand|stood|of|at|total(?:led)?)\s*$|"
                     r"\b(?:booked|sold)\s+(?:so\s+far|to\s+date)?\s*(?:is|was|were|are|stands?\s+at|of)?\s*$", pre[-45:]) and not re.search(r"unsold|unbooked", pre[-45:]):
            return 'booked'
        if re.search(r"unsold|unbooked|remain(?:s|ing)?\s+(?:unsold|unbooked|available|for\s+sale|on\s+offer)|remaining\s+(?:inventory|apartments|units|stock)|"
                     r"still\s+available|left\s+to\s+(?:sell|be\s+sold)|inventory\s+left|yet\s+to\s+be\s+(?:sold|booked)|available\s+(?:for\s+sale|inventory)", pre[-60:]) and \
                not re.search(r"\btotal\b|in\s+all|comprises", pre[-30:]):
            return 'unsold'
        if re.search(r"\bbook(?:ed|ings?)\b|\bsold\b(?!\s+out)|\breserved\b", pre[-60:]) and not re.search(r"\btotal\b|in\s+all|comprises|inventory", pre[-40:]):
            return 'booked'
        if re.search(r"(?:every|each|per)\s+floor(?:\s+(?:of|in)\s+(?:each|every|the|a)\s+tower)?\s+(?:carries|has|holds|houses|contains|accommodates|offers|with)?\s*$", pre[-50:]):
            return 'per_floor'
        return 'total'
    lab = re.sub(r"(?:\b(?:in|of|at|for)\s+)?(?:mainproject|otherproject[a-z])", " ", pre[-90:])
    lab = re.sub(r"[,()]", " ", lab)
    lab = re.sub(r"\s+", " ", lab)[-75:]
    cues = [
        ('per_floor', r"(?:apartments?|units?|homes|flats)\s+(?:on|per)\s+(?:every|each|a)?\s*floor(?:\s+(?:of|in)\s+(?:each|every|the|a)\s+tower)?|per\s+floor|on\s+(?:every|each)\s+floor(?:\s+(?:of|in)\s+(?:each|every|the|a)\s+tower)?|"
                      r"(?:every|each)\s+floor(?:\s+(?:of|in)\s+(?:each|every|the|a)\s+tower)?\s+(?:carries|has|holds|houses|contains|accommodates|offers)"),
        ('unsold', r"unsold|unbooked|yet\s+to\s+be\s+(?:sold|booked)|remaining\s+(?:inventory|apartments|units)|available\s+(?:apartments|units)|"
                   r"left\s+to\s+sell|inventory\s+left|still\s+(?:available|on\s+offer)"),
        ('booked', r"\bbooked\b|\bbookings?\b|(?<!un)\bsold\b(?!\s+out)|\breserved\b"),
        ('floors', r"floors?\s+(?:in|per|of)\s+(?:each|every|a)\s+tower|(?:residential\s+)?floors?\b(?!\s+of\s+(?:each|every)\s+tower)|storeys"),
        ('towers', r"(?<!each\s)(?<!every\s)(?<!per\s)(?<!a\s)\btowers?\b"),
        ('total', r"\btotal\b|number\s+of\s+apartments|inventory\s+of|comprises|apartments\s+in\s+the\s+project|full\s+inventory|apartments?\s+(?:count|planned)"),
        ('size', r"(?:apartment|unit|flat|home)\s+size|\bsize\b|measures?|carpet\s+area|built[\s-]up|super\s+area|sq(?:uare)?\.?\s*(?:ft|feet)"),
    ]
    prio = {r: k for k, (r, _) in enumerate(cues)}
    best = None
    for role, pat in cues:
        ends = [mm.end() for mm in re.finditer(pat, lab)]
        if ends and (best is None or max(ends) > best[0] or (max(ends) == best[0] and prio[role] < prio[best[1]])):
            best = (max(ends), role)
    if best is None:
        if re.search(r"apartments\b[^.]{0,25}$", lab):
            return 'total'
        return None
    role = best[1]
    if role == 'size' and re.search(r"club|amenit|land|plot|retail|commercial", lab[-75:]):
        return None
    if role == 'booked' and re.search(r"sold\s+out", lab):
        return None
    return role

# Extract apartment supply, bookings and current price
def extract_re(text):
    text = normalize_text(pre_normalize(text))
    names = []
    for m in PROJ_RE.finditer(text):
        if m.group(0) not in names:
            names.append(m.group(0))
    main_name = names[0] if names else None
    other_names = [n for n in names[1:] if n.split()[0] != (main_name or ' ').split()[0]]
    for k, n in enumerate(other_names):
        text = text.replace(n, f'OtherProject{chr(97 + k)}')
    if main_name:
        text = text.replace(main_name, 'MainProject')
    other_names = [f'OtherProject{chr(97 + k)}' for k in range(len(other_names))]
    main_name = 'MainProject' if main_name else None

    qs = []
    sec_other = False
    q_other = False
    last_other = False
    last_orig_money = None
    orig_role = None
    role_hint = None
    last_q = None
    last_q_sent = -9

    for sent_idx, (kind, s, _, _, _) in enumerate(report_units(text)):
        if kind == 'heading':
            orig_role = None
            sec_other = (bool(PROPERTY_OTHER_RE.search(s)) or any(n in s for n in other_names) or bool(re.search(r"\b(?:next|following|later|new)\s+phase\b", s, re.I)) or
                         bool(re.search(r"\b(?:sister|neighbou?r|comparison|rival|competitor|phase\s*(?:2|ii|two)|second\s+phase|track\s+record|previous)\b", s, re.I))) \
                and not (main_name and main_name in s and not any(n in s for n in other_names))
            continue
        if kind == 'q':
            q_other = bool(re.search(r"\b(?:second\s+phase|phase\s*(?:2|ii)|sister|neighbou?r|rival|competitor|previous\s+project|other\s+project|otherproject)", s, re.I))
            continue
        if kind != 'a':
            q_other = False
        sent_other = sec_other or (kind == 'a' and q_other)
        if kind != 'table' and last_other and re.search(r"\bthere\b|\bthat\s+(?:project|phase|development)\b|\bthe\s+sister\b", s, re.I) and not (main_name and main_name in s):
            sent_other = True
        if kind != 'table' and last_other and re.match(
                r"^\s*(?:its|their|it|there|of\s+its|of\s+their|of\s+these|of\s+them|of\s+those|in\s+that\s+(?:second\s+)?phase|that\s+phase|this\s+phase|the\s+phase)\b", s, re.I):
            sent_other = True
        if kind == 'table' and not sec_other:
            sent_other = False
        _cls = clauses(s)
        _off = 0
        for cl in _cls:
            _cpos = s.find(cl, _off)
            sent_prefix = s[:_cpos] if _cpos >= 0 else ''
            _off = max(_off, _cpos + len(cl)) if _cpos >= 0 else _off
            other = sent_other
            if PROPERTY_OTHER_RE.search(cl) or any(n in cl for n in other_names):
                other = True
                sent_other = True
            elif main_name and main_name in cl and not sec_other and re.search(r"^\W*(?:\w+\W+){0,8}?" + main_name, cl) and \
                    not re.search(r"(?:than|rather\s+than|not|unlike|versus|compared\s+(?:with|to)|\bto)\s+(?:the\s+|in\s+)?" + main_name, cl):
                other = False
                sent_other = False
            if other:
                continue
            money = find_all_money(cl)
            mspans = [(a, b) for a, b, _, _ in money]
            for k, (a, b, v, _) in enumerate(money):
                left = cl[(money[k - 1][1] if k > 0 else 0):a]
                right = cl[b:(money[k + 1][0] if k + 1 < len(money) else len(cl))]
                pre, post = left[-110:], right[:70]
                if SPEND_RE.search(pre[-60:]) and not re.search(r"price|rate|per\s+(?:unit|apartment|sq)", pre[-40:] + post[:25], re.I):
                    continue
                orig, revised = orig_revised(cl[max(0, a - 110):a], post)
                wide = (sent_prefix + cl[:a])[-200:]
                _post_rate = re.match(r"^\s*(?:per|/|a)\s*(?:sq(?:uare)?\.?\s*(?:ft|feet|foot|metre)|sq\s*ft|square)", post, re.I)
                _post_unit = re.match(r"^\s*(?:per|a|for\s+an?|each)\s*(?:unit|apartment|flat|home)", post, re.I)
                _rc = max((mm.end() for mm in re.finditer(r"per[\s-]+(?:sq(?:uare)?\.?[\s-]*(?:ft|feet|foot)|square[\s-]+f)|/\s*sq|\brates?\b", wide, re.I)), default=-1)
                _pc = max((mm.end() for mm in re.finditer(r"price|priced|\bcost|paying|ticket|per\s+(?:unit|apartment)|average\s+apartment|an\s+apartment", wide, re.I)), default=-1)
                if _post_rate:
                    is_rate, is_price = True, False
                elif _post_unit:
                    is_rate, is_price = False, True
                else:
                    is_rate = _rc >= 0 and _rc > _pc
                    is_price = _pc >= 0 and not is_rate
                _p = re.split(r"[,;(]|\bwhich\b|\bwell\b|\bwhereas\b|\bwhile\b|\bbut\b|\bcompared\b|\bagainst\b|\bthan\b", post[:35])[0]
                rl, rcur = LAUNCH_RE.search(_p), CURRENT_RE.search(_p)
                if rl or rcur:
                    launch = bool(rl) and (not rcur or rl.start() < rcur.start())
                elif re.match(r"^\s*,?\s*(?:an?\s+|the\s+|which\s+(?:was|is)\s+(?:the\s+)?)?(?:\w+\s+)?(?:earlier\s+and\s+lower|earlier|launch[\s-](?:time\s+)?(?:price|rate|figure|level|average))\b", right[:80], re.I):
                    launch = True
                else:
                    whole_pre = sent_prefix + cl[:a]
                    lp = max((mm.end() for mm in LAUNCH_RE.finditer(whole_pre)), default=-1)
                    cp = max((mm.end() for mm in CURRENT_RE.finditer(whole_pre)), default=-1)
                    launch = lp > cp
                if revised and launch and not rl and not LAUNCH_RE.search(pre[-45:]):
                    launch = False
                current = (not launch) and bool(CURRENT_RE.search(cl[:a][-90:] + ' ' + post[:45]))
                role_m = 'rate' if is_rate else 'price' if is_price else None
                if role_m is None and last_orig_money is not None and (REV_CUE.search(pre[-80:]) or re.search(r"\b(?:replaces|supersedes)\b", post[:40], re.I)):
                    role_m, revised, launch, current = last_orig_money, True, False, True
                if role_m is not None:
                    if orig:
                        last_orig_money = role_m
                    qs.append(Quantity(role=role_m, value=v, orig=orig, revised=revised, launch=launch, current=current))

            for m in re.finditer(r"(\d+(?:\.\d+)?)\s*%", cl):
                if any(a <= m.start() < b for a, b in mspans):
                    continue
                pre = cl[max(0, m.start() - 80):m.start()].lower()
                post = cl[m.end():m.end() + 100].lower()
                ctx = pre[-60:] + ' ' + re.split(r"[;]|\bwhile\b|\bwhereas\b", post)[0][:90]
                if re.search(r"booked|sold|reserved|bookings?|taken\s+up", ctx) and not re.search(r"sold\s+out", ctx):
                    orig, revised = orig_revised(pre, post)
                    if re.search(r"unsold|unbooked|remain(?:s|ing)?\s+(?:unsold|unbooked|available|for\s+sale|on\s+offer|to\s+be\s+sold)|remaining\s+(?:inventory|apartments|units|stock)|still\s+available", pre[-40:] + post[:60]):
                        qs.append(Quantity(role='unsold_pct', value=float(m.group(1)), orig=orig, revised=revised))
                    else:
                        qs.append(Quantity(role='booked_pct', value=float(m.group(1)), orig=orig, revised=revised))
                elif re.search(r"unsold|unbooked", ctx):
                    qs.append(Quantity(role='unsold_pct', value=float(m.group(1))))
                elif re.search(r"rose|risen|rise|increase|up|higher|appreciat|climb|grew|jump|gain", pre[-70:] + post[:40]) and \
                        re.search(r"price|rate|launch|value", pre[-90:] + post[:60]):
                    qs.append(Quantity(role='price_growth', value=float(m.group(1))))

            for m in NUM_RE.finditer(cl):
                if any(a <= m.start() < b for a, b in mspans):
                    continue
                v = num(m)
                if 1990 <= v <= 2040 and ',' not in m.group(0) and \
                        not re.match(r"^\s*(?:apartments?|units?|homes|flats|sq)", cl[m.end():m.end() + 12], re.I):
                    continue
                if re.search(r"(?:RERA|registration|number|no\.?)\s*[A-Z]*$", cl[max(0, m.start() - 25):m.start()], re.I) and v > 1e6:
                    continue
                role = role_count(cl, m)
                pre90 = cl[max(0, m.start() - 90):m.start()]
                if role is None and role_hint and re.match(r"^\s*(?:it|this|that|the\s+(?:figure|number|count))\b", s, re.I):
                    role = role_hint
                if last_q is not None and last_q_sent == sent_idx - 1 and orig_role is None and REVISED_TO.search(pre90[-60:]) and \
                        re.match(r"^\s*(?:that|this|it|the)\s*(?:number|figure|count|total)?\b", s, re.I) and not last_q.orig and last_q.role in ('total', 'booked', 'unsold'):
                    last_q.orig = True
                    orig_role = last_q.role
                if orig_role is not None and REVISED_TO.search(pre90[-60:]) and (role is None or re.search(r"(?:was|been|has\s+been|it\s+was)\s+(?:later\s+|since\s+|subsequently\s+)?revised\s+to\s*$|revised\s+(?:it|this|that|the\s+(?:figure|number|count))\s+to\s*$", pre90[-50:], re.I)):
                    role = orig_role
                if role is None and orig_role is not None and re.match(r"^\s*(?:\w+\s+){0,2}(?:replaces|supersedes)\b", cl[m.end():m.end() + 40], re.I):
                    role = orig_role
                if role is None:
                    continue
                orig, revised = orig_revised(pre90, cl[m.end():m.end() + 60])
                if orig:
                    orig_role = role
                elif role == orig_role and REV_CUE.search(pre90[-60:]):
                    revised = True
                last_q = Quantity(role=role, value=v, orig=orig, revised=revised)
                last_q_sent = sent_idx
                qs.append(last_q)

        last_other = sent_other
        if not re.search(r"\d", s):
            role_hint = ('total' if re.search(r"total\s+(?:number\s+of\s+)?apartments|apartment\s+total|total\s+(?:apartment|unit)\s+count", s, re.I) else
                         'unsold' if re.search(r"unsold", s, re.I) else 'booked' if re.search(r"booked|bookings?", s, re.I) else None)
        else:
            role_hint = None
    return qs

def parse_property(text):
    qs = extract_re(text)
    total = pick_quantity(qs, 'total')
    booked = pick_quantity(qs, 'booked')
    unsold = pick_quantity(qs, 'unsold')
    dimensions = [pick_quantity(qs, role) for role in ('towers', 'floors', 'per_floor')]
    revised_total = any(q.revised and not q.orig for q in qs if q.role == 'total')
    if all(value is not None and value > 0 for value in dimensions) and not revised_total:
        inventory = float(np.prod(dimensions))
        if inventory >= max(booked or 0, unsold or 0):
            total = inventory
    if booked is None:
        bp = pick_quantity(qs, 'booked_pct')
        if bp is not None and total:
            booked = total * bp / 100
    un = unsold
    if booked is None and total:
        if un is not None:
            booked = total - un
        else:
            up = pick_quantity(qs, 'unsold_pct')
            if up is not None:
                booked = total * (1 - up / 100)
    if total is None and un is not None and booked is not None:
        total = booked + un
    price = pick_quantity(qs, 'price', prefer_current=True)
    if price is None:
        rate, size = pick_quantity(qs, 'rate', prefer_current=True), pick_quantity(qs, 'size')
        if rate and size:
            price = rate * size
    if price is None:
        g, size = pick_quantity(qs, 'price_growth'), pick_quantity(qs, 'size')
        lp = [q for q in qs if q.role == 'price' and q.launch and not q.orig]
        lr = [q for q in qs if q.role == 'rate' and q.launch and not q.orig]
        if g is not None and lp:
            price = lp[-1].value * (1 + g / 100)
        elif g is not None and lr and size:
            price = lr[-1].value * size * (1 + g / 100)
    x = np.array([np.nan if total is None else total, np.nan if booked is None else booked,
                  np.nan if price is None else price], float)
    return x


## Route reports and extract inputs
The rule-based router chooses a parser for each report. A TF–IDF logistic regression trained on the training routes flags test reports for review if the two methods disagree.


In [7]:
def report_type(text):
    lowered = text.lower()
    for term, domain in (("inpatient", "Hospital"), ("quintal", "Farmer"), ("apartment", "Real Estate")):
        if term in lowered:
            return domain
    return "Corporate"


PARSERS = {"Hospital": parse_hospital, "Farmer": parse_farmer,
           "Real Estate": parse_property, "Corporate": parse_corporate}


def extract_reports(frame):
    domains = frame.text.map(report_type).to_numpy()
    inputs = []
    recoveries = 0
    for text, domain in zip(frame.text, domains):
        values = PARSERS[domain](text)
        if domain == "Hospital" and not np.isfinite(values).all():
            recoveries += 1
            values = recover_hospital(text, values)
        if not np.isfinite(values).all():
            raise ValueError(f"Incomplete {domain} inputs in report {frame.id.iloc[len(inputs)]}")
        inputs.append(values)
    print(f"Parsed {len(frame):,} reports. Hospital recovery calls = {recoveries}.")
    return domains, inputs


train_domains, train_inputs = extract_reports(train)
test_domains, test_inputs = extract_reports(test)
targets = train.target.to_numpy(dtype=float)

router = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=30000),
                       LogisticRegression(max_iter=500, random_state=SEED))
routes = router.fit(train.text, train_domains).predict(test.text)
mismatch = routes != test_domains
print(f"Test routing disagreements = {mismatch.sum()}")
if mismatch.any():
    print(pd.DataFrame({"id": test.id[mismatch], "parser": test_domains[mismatch],
                        "classifier": routes[mismatch]}).to_string(index=False))


Parsed 1,195 reports. Hospital recovery calls = 0.
Parsed 1,194 reports. Hospital recovery calls = 0.
Test routing disagreements = 0


## Fit each domain formula
Use all labeled reports. Hospital uses a log-linear power law. The other domains use monetary feature combinations and relative residuals with a scale floor near zero. Hospital and Real Estate use a three-robust-scale trimmed fit, Farmer uses Tukey biweight, and Corporate uses Cauchy weights. Iteration stops at a relative coefficient change of $10^{-9}$ or 50 updates.


In [8]:
def formula_inputs(x, domain):
    if domain == "Hospital":
        return np.column_stack([np.ones(len(x)), np.log(x[:, 0] / 1e9), -np.log(x[:, 1] / 5e4)])
    if domain == "Farmer":
        return np.column_stack([x[:, 0] * x[:, 1], -x[:, 2]]) / 1000
    if domain == "Real Estate":
        return np.column_stack([x[:, 0] * x[:, 2], -x[:, 1] * x[:, 2]]) / 1e7
    return x  # Corporate operating profits are already in crore.



def fit_formula(x, target, domain):
    design = formula_inputs(x, domain)
    hospital = domain == "Hospital"
    y = np.log(target) if hospital else target
    bounds = ([-np.inf, .01, .01], [np.inf, 3, 3]) if hospital else (1e-9, np.inf)
    scale = np.ones_like(y) if hospital else np.maximum(abs(y), .5)

    def solve(weights):
        return lsq_linear(design * weights[:, None], y * weights, bounds=bounds).x

    theta = solve(1 / scale)
    for _ in range(50):
        fitted = design @ theta
        if not hospital:
            scale = np.maximum(abs(fitted), .5)
        residual = (fitted - y) / scale
        median = np.median(residual)
        spread = max(1e-4, 1.4826 * np.median(abs(residual - median)))
        if domain == "Corporate":
            weights = 1 / (1 + (residual / (2.385 * spread))**2)
        elif domain == "Farmer":
            weights = np.maximum(0, 1 - (residual / (4.685 * spread))**2)**2
        else:
            weights = abs(residual - median) <= 3 * spread
        update = solve(np.sqrt(weights) / scale)
        if np.max(abs(update - theta) / np.maximum(1, abs(theta))) <= 1e-9:
            return update
        theta = update
    raise RuntimeError(f"{domain} fit did not converge in 50 iterations")

predictions = np.empty(len(test))
for domain in DOMAINS:
    rows = np.flatnonzero(train_domains == domain)
    query = np.flatnonzero(test_domains == domain)
    if len(query) == 0:
        continue
    theta = fit_formula(np.stack([train_inputs[i] for i in rows]), targets[rows], domain)
    fitted = formula_inputs(np.stack([test_inputs[i] for i in query]), domain) @ theta
    predictions[query] = np.exp(fitted) if domain == "Hospital" else fitted


## Write the Kaggle submission
Preserve the test-file order and leading zeros in each ID. The saved CSV contains one four-decimal target per report.


In [9]:
submission = pd.DataFrame({"id": test.id, "target": predictions})
assert len(submission) == 1194 and np.isfinite(predictions).all()
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_FILE, index=False, float_format="%.4f", lineterminator="\r\n")
saved = pd.read_csv(OUTPUT_FILE, dtype=str)
assert list(saved) == ["id", "target"] and saved.id.equals(test.id)
assert saved.target.str.fullmatch(r"-?\d+\.\d{4}").all()
print(f"Wrote {len(saved):,} rows to {OUTPUT_FILE}.")


Wrote 1,194 rows to report_reader_outputs\submission.csv.
